# Armenian dubbing: Colab test and ablation (English to Eastern Armenian)

This notebook measures the current pipeline (baseline) against each proposed change, one change at a time, then runs two combined rows. Background and sources: [`docs/research/2026-10-dubbing-audit-and-plan.md`](../docs/research/2026-10-dubbing-audit-and-plan.md).

**How to run**
1. Runtime > Change runtime type > GPU. Free = T4. Paid = L4 or A100.
2. Add Colab secrets (key icon): `HF_TOKEN` (required). Optional: `GEMINI_API_KEY`, `AZURE_SPEECH_KEY`, `AZURE_REGION`, `FISH_API_KEY`.
3. On Hugging Face, accept the terms of `pyannote/speaker-diarization-community-1`, `google/translategemma-*`, `google/gemma-3-*`, and `openlanguagedata/flores_plus`.
4. Set `PROFILE` in the next cell, then Run all. Every stage is cached on Drive, so after a disconnect, Run all resumes where it stopped.

**Licenses.** Rows tagged `[NC]` use non-commercial weights and are research references only. `edge-tts` has no commercial terms; here it is a free stand-in for Azure `hy-AM` voices (same voice family).

Heavy models run in separate virtual environments (`uv`), so their pinned dependencies do not break each other.

In [ ]:
# ---- 1. Settings -------------------------------------------------------------
PROFILE = "free_t4"   # "free_t4" | "paid_l4" | "paid_a100"
RUN = dict(asr=True, mt=True, tts=True, video=True, lipsync=True, api_rows=False)
ROW_FILTER = None     # e.g. ["baseline", "+ timing fixes"] to run only some ablation rows
USE_DRIVE = True      # cache on Google Drive (Free sessions get killed)
MY_CLIPS_DIR = "/content/drive/MyDrive/dub_clips"   # optional: your own TV clips (.mp4/.mov/.mkv)
REPO_URL = "https://github.com/Edmon02/armenian-video-dubbing.git"
GEMINI_MODEL = "gemini-flash-latest"   # unverified alias: set the exact Gemini model id you use
FISH_MODEL = "s2.1-pro"                # Fish Audio backend name from their docs (unverified in SDK)
SEED = 7

import os
def _secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name) or ""
    except Exception:
        return os.environ.get(name, "")

HF_TOKEN = _secret("HF_TOKEN")
GEMINI_API_KEY = _secret("GEMINI_API_KEY")
AZURE_SPEECH_KEY, AZURE_REGION = _secret("AZURE_SPEECH_KEY"), _secret("AZURE_REGION")
FISH_API_KEY = _secret("FISH_API_KEY")

PROFILES = {
    # llm_quant: how 4B-12B LLMs are loaded. T4 has no bf16, and Gemma/T5 overflow in fp16,
    # so on T4 we use 4-bit weights with fp32 compute.
    "free_t4": dict(n_per_bucket=4, n_hdtf=3, max_clip_sec=15, repo_profile="configs/profiles/colab_t4_demo.yaml",
                    whisperx_model="large-v3-turbo", whisperx_compute="float16", whisperx_batch=8,
                    tgemma="google/translategemma-4b-it", rewrite_llm="google/gemma-3-4b-it",
                    madlad="google/madlad400-3b-mt", llm_quant="nf4", omni_steps=16,
                    lipsync=["musetalk15", "latentsync15"], n_centroid=16),
    "paid_l4": dict(n_per_bucket=8, n_hdtf=6, max_clip_sec=30, repo_profile="configs/profiles/lightning_l4_demo.yaml",
                    whisperx_model="large-v3-turbo", whisperx_compute="float16", whisperx_batch=16,
                    tgemma="google/translategemma-12b-it", rewrite_llm="google/gemma-3-12b-it",
                    madlad="google/madlad400-10b-mt", llm_quant="int8", omni_steps=32,
                    lipsync=["musetalk15", "latentsync16"], n_centroid=24),
    "paid_a100": dict(n_per_bucket=14, n_hdtf=8, max_clip_sec=60, repo_profile="configs/profiles/lightning_a100_80gb_full.yaml",
                      whisperx_model="large-v3-turbo", whisperx_compute="float16", whisperx_batch=16,
                      tgemma="google/translategemma-12b-it", rewrite_llm="google/gemma-3-12b-it",
                      madlad="google/madlad400-10b-mt", llm_quant="none", omni_steps=32,
                      lipsync=["musetalk15", "latentsync16"], n_centroid=32),
}
P = PROFILES[PROFILE]
print(PROFILE, P)

In [ ]:
# ---- 2. GPU check, folders ------------------------------------------------------
import subprocess, json, sys, time, shutil
from pathlib import Path
import torch

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
os.environ["TOKENIZERS_PARALLELISM"] = "false"

assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > GPU"
props = torch.cuda.get_device_properties(0)
GPU_NAME, VRAM_GB = props.name, props.total_memory / 1e9
BF16 = torch.cuda.is_bf16_supported()
print(f"GPU: {GPU_NAME}  VRAM: {VRAM_GB:.1f} GB  compute {props.major}.{props.minor}  bf16: {BF16}")
if PROFILE == "free_t4" and VRAM_GB > 20:
    print("Note: free_t4 profile on a bigger GPU. Results are valid, but the fit matrix below is for T4.")
if PROFILE != "free_t4" and VRAM_GB < 20:
    print("WARNING: paid profile on a <20 GB GPU. LatentSync 1.6 and 12B LLMs will fail or offload to CPU.")
if not HF_TOKEN:
    print("WARNING: HF_TOKEN missing. pyannote, Gemma, and FLORES+ downloads will fail.")

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/dub_ablation"
else:
    WORK = "/content/dub_ablation"
LOCAL, VENVS = "/content/local", "/content/venvs"
WORKERS = f"{WORK}/workers"
for d in [f"{WORK}/{x}" for x in ("cache", "data", "out", "workers", "results")] + [LOCAL, VENVS]:
    os.makedirs(d, exist_ok=True)
print("WORK =", WORK)

In [ ]:
# ---- 3. Model fit matrix for this GPU ----------------------------------------------
# VRAM = approx. inference peak at the precision shown. Sources: model cards and READMEs in the
# report; values marked "U" are estimates (unverified). headroom = 85% of the card.
import pandas as pd
MODELS = [
    # name, stage, vram_gb, needs_bf16, commercial_ok, note
    ("Whisper small, 4-bit (baseline ASR, free)", "ASR", 1.0, False, True, "MIT"),
    ("Whisper large-v3, 4-bit (baseline ASR, paid)", "ASR", 2.5, False, True, "MIT"),
    ("WhisperX large-v3-turbo fp16 + align", "ASR", 4.0, False, True, "MIT/BSD-2, U"),
    ("Parakeet-TDT-0.6b-v2", "ASR", 3.0, False, True, "CC-BY-4.0, U"),
    ("pyannote community-1", "Diarization", 1.5, False, True, "CC-BY-4.0 gated, U"),
    ("SeamlessM4T v2 large fp16 (baseline MT)", "MT", 6.0, False, False, "CC-BY-NC"),
    ("TranslateGemma-4B nf4", "MT", 4.0, False, True, "Gemma terms, U"),
    ("TranslateGemma-12B nf4", "MT", 9.0, False, True, "Gemma terms, U"),
    ("TranslateGemma-12B int8", "MT", 14.0, False, True, "Gemma terms, U"),
    ("TranslateGemma-12B bf16", "MT", 26.0, True, True, "Gemma terms"),
    ("MADLAD-400-3B nf4", "MT", 3.0, False, True, "Apache-2.0, U"),
    ("MADLAD-400-10B bf16", "MT", 22.0, True, True, "Apache-2.0, U"),
    ("Gemma-3-4B-it nf4 (rewrite)", "MT", 4.0, False, True, "Gemma terms, U"),
    ("Gemma-3-12B-it int8 (rewrite)", "MT", 14.0, False, True, "Gemma terms, U"),
    ("edge-tts (baseline TTS, cloud)", "TTS", 0.0, False, False, "no commercial terms"),
    ("OpenVoice v2 converter", "VC", 1.5, False, True, "MIT, U"),
    ("OmniVoice 0.6B fp16", "TTS", 4.0, False, False, "weights CC-BY-NC, U"),
    ("Demucs htdemucs", "Mix", 3.0, False, True, "MIT, U"),
    ("MuseTalk 1.5 fp16", "LipSync", 4.0, False, True, "MIT; authors ran it on 4 GB"),
    ("LatentSync 1.5", "LipSync", 8.0, False, True, "Apache-2.0; README min VRAM"),
    ("LatentSync 1.6", "LipSync", 18.0, False, True, "Apache-2.0; README min VRAM"),
    ("InfiniteTalk (Wan2.1-14B)", "LipSync", 40.0, True, True, "Apache-2.0; VRAM not published (U). Not run here"),
    ("SyncNet v2 (LSE eval)", "Metric", 1.0, False, True, "MIT"),
    ("COMET-22 (eval)", "Metric", 3.0, False, True, "Apache-2.0"),
    ("UTMOSv2 (eval)", "Metric", 2.0, False, True, "U"),
    ("WavLM-SV + ECAPA + VoxLingua LID (eval)", "Metric", 1.5, False, True, "MIT/Apache"),
    ("NVIDIA hy FastConformer (eval back-ASR)", "Metric", 2.0, False, True, "CC-BY-4.0"),
]
def fit(v, bf16):
    if bf16 and not BF16: return "no (needs bf16)"
    return "fits" if v <= 0.85 * VRAM_GB else "no (VRAM)"
FIT = pd.DataFrame(MODELS, columns=["model", "stage", "vram_gb", "needs_bf16", "commercial_ok", "note"])
FIT["this_gpu"] = [fit(v, b) for v, b in zip(FIT.vram_gb, FIT.needs_bf16)]
FIT["T4_16GB"] = ["no (needs bf16)" if b else ("fits" if v <= 13.0 else "no (VRAM)") for v, b in zip(FIT.vram_gb, FIT.needs_bf16)]
FIT["L4_24GB"] = ["fits" if v <= 20.0 else "no (VRAM)" for v in FIT.vram_gb]
FIT["A100_40/80GB"] = ["fits" if v <= 68.0 else "no (VRAM)" for v in FIT.vram_gb]
pd.set_option("display.max_colwidth", 60)
display(FIT)

In [ ]:
# ---- 4. Install the main environment (metrics, MT, baseline repo) -------------------------------
# Torch is pinned to the Colab version so pip does not replace it.
!apt-get -qq update > /dev/null && apt-get -qq install -y ffmpeg rubberband-cli sox git-lfs > /dev/null
!pip -q install -U uv
import torch, torchvision, torchaudio
Path(f"{LOCAL}/constraints.txt").write_text(
    f"torch=={torch.__version__}\ntorchvision=={torchvision.__version__}\ntorchaudio=={torchaudio.__version__}\n")
PKGS = ('sacrebleu unbabel-comet jiwer speechbrain bitsandbytes accelerate edge-tts pyloudnorm librosa soundfile demucs '
        'pandas pyarrow yt-dlp loguru "torchmetrics[image]" torch-fidelity google-genai azure-cognitiveservices-speech '
        'fish-audio-sdk opencv-python-headless "transformers>=4.57" huggingface_hub tabulate')
!uv pip install --system -q -c {LOCAL}/constraints.txt {PKGS}
!uv pip install --system -q -c {LOCAL}/constraints.txt --no-deps facenet-pytorch
!uv pip install --system -q -c {LOCAL}/constraints.txt git+https://github.com/sarulab-speech/UTMOSv2.git || echo "UTMOSv2 install failed; MOS column will be empty"

REPO = "/content/armenian-video-dubbing"
if not os.path.exists(REPO):
    !git clone -q --depth 1 {REPO_URL} {REPO}
sys.path.insert(0, REPO)
rb = subprocess.run("rubberband --help 2>&1", shell=True, capture_output=True, text=True).stdout
RB_ENGINE = "--fine" if "--fine" in rb else "--crisp 3"   # R3 engine needs rubberband >= 3.0
print("rubberband engine flag:", RB_ENGINE)

In [ ]:
# ---- 5. Helpers: shell, GPU poller, stage timer, cache, workers, audio, text ----------------------
import gc, re, math, hashlib, threading, tempfile, random, io, unicodedata, asyncio
import numpy as np, soundfile as sf, librosa

def sh(cmd, check=True, cwd=None, env=None, timeout=None):
    r = subprocess.run(cmd, shell=True, cwd=cwd, env=env, capture_output=True, text=True, timeout=timeout)
    if check and r.returncode != 0:
        raise RuntimeError(f"command failed ({r.returncode}): {cmd}\n{(r.stdout + r.stderr)[-3000:]}")
    return r

def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

class GpuPoller:
    """Peak device memory via nvidia-smi. Covers subprocess workers too."""
    def __init__(self, every=0.5):
        self.every, self.peak, self._stop = every, 0.0, threading.Event()
    def _run(self):
        while not self._stop.is_set():
            try:
                out = subprocess.run("nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits",
                                     shell=True, capture_output=True, text=True).stdout
                self.peak = max(self.peak, float(out.strip().splitlines()[0]) / 1024)
            except Exception:
                pass
            time.sleep(self.every)
    def __enter__(self):
        self._t = threading.Thread(target=self._run, daemon=True); self._t.start(); return self
    def __exit__(self, *a):
        self._stop.set(); self._t.join(timeout=2)

PERF, STAGE_PERF = [], {}

class stage:
    """Times one stage; records torch peak (this process) and nvidia-smi peak (all processes)."""
    def __init__(self, name, variant, audio_sec=None):
        self.row = dict(stage=name, variant=variant, audio_sec=audio_sec)
    def __enter__(self):
        free_gpu(); torch.cuda.reset_peak_memory_stats()
        self.p = GpuPoller().__enter__(); self.t = time.time(); return self.row
    def __exit__(self, *exc):
        self.p.__exit__()
        self.row.update(seconds=time.time() - self.t, torch_peak_gb=torch.cuda.max_memory_allocated() / 1e9,
                        smi_peak_gb=self.p.peak)
        PERF.append(self.row)

def _h(s): return hashlib.sha1(s.encode()).hexdigest()[:12]

def cached_stage(name, variant, key, fn, audio_sec=None):
    """Run fn() once per (name, variant, key); store result + perf on Drive."""
    f = Path(f"{WORK}/cache/{name}_{variant}_{_h(key)}.json")
    if f.exists():
        d = json.load(open(f)); STAGE_PERF[f.name] = d["perf"]; PERF.append({**d["perf"], "cached": True})
        return d["result"], f.name
    with stage(name, variant, audio_sec) as row:
        res = fn()
    if isinstance(res, dict) and "_worker_peak_gb" in res:
        row["torch_peak_gb"] = max(row["torch_peak_gb"], res.pop("_worker_peak_gb"))
    json.dump({"result": res, "perf": row}, open(f, "w"), ensure_ascii=False)
    STAGE_PERF[f.name] = row
    return res, f.name

def run_worker(venv_name, script, job, cwd=None, timeout=7200):
    """Run a worker script inside its own venv. Job and result are JSON files."""
    jf = tempfile.mktemp(suffix=".json", dir=LOCAL); of = jf.replace(".json", ".out.json")
    json.dump(job, open(jf, "w"), ensure_ascii=False)
    env = {k: v for k, v in os.environ.items() if k not in ("PYTHONPATH", "PYTHONHOME", "MPLBACKEND")}
    r = subprocess.run([f"{VENVS}/{venv_name}/bin/python", f"{WORKERS}/{script}", jf, of], cwd=cwd,
                       capture_output=True, text=True, timeout=timeout, env=env)
    if r.returncode != 0 or not os.path.exists(of):
        raise RuntimeError(f"{script} failed:\n{(r.stdout + r.stderr)[-4000:]}")
    return json.load(open(of))

def load(p, sr=16000):
    y, _ = librosa.load(p, sr=sr, mono=True); return y.astype(np.float32)

def save(p, y, sr):
    Path(p).parent.mkdir(parents=True, exist_ok=True); sf.write(p, np.asarray(y, np.float32), sr); return p

def trim(y, sr, top_db=35):
    if len(y) == 0: return y
    yt, _ = librosa.effects.trim(y, top_db=top_db); return yt

def dur(p): return sf.info(p).duration

# Armenian: syllables = vowel letters, with "ու" as one vowel and "և" as "եվ". Unwritten schwas are not
# counted; the TTS speaking-rate calibration absorbs that bias.
HY_CHARS = re.compile(r"[\u0531-\u0556\u0561-\u0587]")
def hy_syllables(t):
    t = t.lower().replace("ու", "u").replace("և", "եվ")
    return sum(c in "աեէըիոօu" for c in t)
def hy_ratio(t):
    letters = [c for c in t if c.isalpha()]
    return sum(bool(HY_CHARS.match(c)) for c in letters) / max(1, len(letters))
def en_syllables(t):
    return sum(max(1, len(re.findall(r"[aeiouy]+", w))) for w in re.findall(r"[a-z']+", t.lower()))
def norm_text(t):
    t = unicodedata.normalize("NFKC", t).lower()
    t = re.sub(r"[^\w\s']", " ", t)
    return re.sub(r"\s+", " ", t).strip()
print("helpers ready")

In [ ]:
# ---- 6. Worker scripts (each runs inside its own venv) -------------------------------------------
W = {}
W["asr_whisperx.py"] = """
import json, sys, gc, time, torch, whisperx
job = json.load(open(sys.argv[1])); dev = "cuda"; res = {"files": {}}
t0 = time.time(); torch.cuda.reset_peak_memory_stats()
model = whisperx.load_model(job["model"], dev, compute_type=job["compute_type"], language="en")
audios = {p: whisperx.load_audio(p) for p in job["files"]}
raw = {p: model.transcribe(a, batch_size=job["batch_size"], language="en") for p, a in audios.items()}
del model; gc.collect(); torch.cuda.empty_cache()
am, meta = whisperx.load_align_model(language_code="en", device=dev)
for p, a in audios.items():
    al = whisperx.align(raw[p]["segments"], am, meta, a, dev, return_char_alignments=False)
    segs = []
    for s in al["segments"]:
        words = [{"word": w.get("word", ""), "start": w.get("start"), "end": w.get("end")} for w in s.get("words", [])]
        segs.append({"start": s.get("start"), "end": s.get("end"), "text": s.get("text", "").strip(), "words": words})
    res["files"][p] = segs
res["seconds"] = time.time() - t0; res["_worker_peak_gb"] = torch.cuda.max_memory_allocated() / 1e9
json.dump(res, open(sys.argv[2], "w"))
"""
W["diar_pyannote.py"] = """
import json, sys, time, torch
from pyannote.audio import Pipeline
job = json.load(open(sys.argv[1])); res = {"files": {}}; torch.cuda.reset_peak_memory_stats()
try:
    pipe = Pipeline.from_pretrained(job["model"], token=job["hf_token"])
except TypeError:
    pipe = Pipeline.from_pretrained(job["model"], use_auth_token=job["hf_token"])
pipe.to(torch.device("cuda"))
for p in job["files"]:
    out = pipe(p)
    ann = getattr(out, "exclusive_speaker_diarization", None) or getattr(out, "speaker_diarization", None) or out
    res["files"][p] = [[float(t.start), float(t.end), str(s)] for t, _, s in ann.itertracks(yield_label=True)]
res["_worker_peak_gb"] = torch.cuda.max_memory_allocated() / 1e9
json.dump(res, open(sys.argv[2], "w"))
"""
W["asr_nemo.py"] = """
import json, sys, torch
import nemo.collections.asr as nemo_asr
job = json.load(open(sys.argv[1])); torch.cuda.reset_peak_memory_stats()
m = nemo_asr.models.ASRModel.from_pretrained(job["model"]).cuda().eval()
ts = job.get("timestamps", False)
outs = m.transcribe(job["files"], batch_size=job.get("batch_size", 8), timestamps=ts) if ts else m.transcribe(job["files"], batch_size=job.get("batch_size", 8))
if isinstance(outs, tuple): outs = outs[0]
res = {"files": {}}
for p, o in zip(job["files"], outs):
    item = {"text": o.text if hasattr(o, "text") else str(o)}
    if ts and getattr(o, "timestamp", None):
        item["segments"] = [{"start": s["start"], "end": s["end"], "text": s["segment"]} for s in o.timestamp.get("segment", [])]
        item["words"] = [{"word": w["word"], "start": w["start"], "end": w["end"]} for w in o.timestamp.get("word", [])]
    res["files"][p] = item
res["_worker_peak_gb"] = torch.cuda.max_memory_allocated() / 1e9
json.dump(res, open(sys.argv[2], "w"))
"""
W["tts_omnivoice.py"] = """
import json, sys, time, torch, soundfile as sf
from omnivoice import OmniVoice
job = json.load(open(sys.argv[1])); torch.cuda.reset_peak_memory_stats()
model = OmniVoice.from_pretrained("k2-fsa/OmniVoice", device_map="cuda:0", dtype=torch.float16)
res = {"items": []}
for it in job["items"]:
    kw = dict(text=it["text"], ref_audio=it["ref_audio"], num_step=job.get("num_step", 32))
    if it.get("ref_text"): kw["ref_text"] = it["ref_text"]
    if it.get("duration"): kw["duration"] = float(it["duration"])
    t = time.time()
    try:
        audio = model.generate(**kw)
        sf.write(it["out"], audio[0], 24000); res["items"].append({"out": it["out"], "ok": True, "sec": time.time() - t})
    except Exception as e:
        res["items"].append({"out": it["out"], "ok": False, "error": str(e)[:300]})
res["_worker_peak_gb"] = torch.cuda.max_memory_allocated() / 1e9
json.dump(res, open(sys.argv[2], "w"))
"""
W["vc_openvoice.py"] = """
import json, sys, torch
from huggingface_hub import snapshot_download
from openvoice.api import ToneColorConverter
job = json.load(open(sys.argv[1])); torch.cuda.reset_peak_memory_stats()
ck = snapshot_download("myshell-ai/OpenVoiceV2", allow_patterns=["converter/*"])
conv = ToneColorConverter(f"{ck}/converter/config.json", device="cuda", enable_watermark=False)
conv.load_ckpt(f"{ck}/converter/checkpoint.pth")
cache = {}
def se(p):
    if p not in cache: cache[p] = conv.extract_se([p])
    return cache[p]
res = {"items": []}
for it in job["items"]:
    try:
        conv.convert(audio_src_path=it["src"], src_se=se(it["src"]), tgt_se=se(it["ref"]), output_path=it["out"], tau=job.get("tau", 0.3))
        res["items"].append({"out": it["out"], "ok": True})
    except Exception as e:
        res["items"].append({"out": it["out"], "ok": False, "error": str(e)[:300]})
res["_worker_peak_gb"] = torch.cuda.max_memory_allocated() / 1e9
json.dump(res, open(sys.argv[2], "w"))
"""
for name, src in W.items():
    Path(f"{WORKERS}/{name}").write_text(src.lstrip())

# venv specs: (python version, list of `uv pip install` argument strings)
VENV_SPECS = {
    "whisperx": ("3.11", ["whisperx"]),
    "pyannote": ("3.11", ["'pyannote.audio>=4.0'"]),
    "nemo": ("3.11", ["'nemo_toolkit[asr]'"]),
    "omnivoice": ("3.11", ["omnivoice"]),
    "openvoice": ("3.10", ["torch torchaudio librosa soundfile pydub eng_to_ipa inflect unidecode pypinyin cn2an jieba langid huggingface_hub 'numpy<2'",
                           "--no-deps git+https://github.com/myshell-ai/OpenVoice.git"]),
    "musetalk": ("3.10", ["torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu118",
                          "-r {LOCAL}/MuseTalk/requirements.txt",
                          "mmengine 'mmcv==2.0.1' -f https://download.openmmlab.com/mmcv/dist/cu118/torch2.0/index.html",
                          "'mmdet==3.1.0' 'mmpose==1.1.0'"]),
    "latentsync": ("3.10", ["-r {LOCAL}/LatentSync16/requirements.txt"]),
    "syncnet": ("3.10", ["torch torchvision 'numpy<2' scipy opencv-python python_speech_features 'scenedetect==0.5.6.1'"]),
}
def venv(name):
    root = f"{VENVS}/{name}"
    if os.path.exists(f"{root}/.ready"): return root
    py, installs = VENV_SPECS[name]
    print(f"[venv] creating {name} (one-time per session, can take minutes)...")
    sh(f"uv venv --seed -q --python {py} {root}")
    for spec in installs:
        sh(f"uv pip install -q --python {root}/bin/python {spec.format(LOCAL=LOCAL)}", timeout=3600)
    Path(f"{root}/.ready").touch(); return root

def hf_get(venv_name, repo, filename, local_dir, cwd):
    py = f"{VENVS}/{venv_name}/bin/python"
    sh(f"""{py} -c "from huggingface_hub import hf_hub_download as d; d('{repo}', '{filename}', local_dir='{local_dir}')" """, cwd=cwd)

def ensure_musetalk():
    d = f"{LOCAL}/MuseTalk"
    if not os.path.exists(d): sh(f"git clone -q --depth 1 https://github.com/TMElyralab/MuseTalk {d}")
    venv("musetalk")
    if not os.path.exists(f"{d}/models/musetalkV15/unet.pth"):
        for repo, fn, ld in [("TMElyralab/MuseTalk", "musetalkV15/musetalk.json", "models"), ("TMElyralab/MuseTalk", "musetalkV15/unet.pth", "models"),
                             ("stabilityai/sd-vae-ft-mse", "config.json", "models/sd-vae"), ("stabilityai/sd-vae-ft-mse", "diffusion_pytorch_model.bin", "models/sd-vae"),
                             ("openai/whisper-tiny", "config.json", "models/whisper"), ("openai/whisper-tiny", "pytorch_model.bin", "models/whisper"),
                             ("openai/whisper-tiny", "preprocessor_config.json", "models/whisper"), ("yzd-v/DWPose", "dw-ll_ucoco_384.pth", "models/dwpose"),
                             ("ByteDance/LatentSync", "latentsync_syncnet.pt", "models/syncnet")]:
            hf_get("musetalk", repo, fn, ld, d)
        os.makedirs(f"{d}/models/face-parse-bisent", exist_ok=True)
        sh(f"{VENVS}/musetalk/bin/gdown --id 154JgKpzCPW82qINcVieuPH3fZ2e0P812 -O models/face-parse-bisent/79999_iter.pth", cwd=d)
        sh("curl -sL https://download.pytorch.org/models/resnet18-5c106cde.pth -o models/face-parse-bisent/resnet18-5c106cde.pth", cwd=d)
    return d

def ensure_latentsync(ver):
    d = f"{LOCAL}/LatentSync{ver}"
    if not os.path.exists(d): sh(f"git clone -q --depth 1 https://github.com/bytedance/LatentSync {d}")
    if not os.path.exists(f"{LOCAL}/LatentSync16"): sh(f"git clone -q --depth 1 https://github.com/bytedance/LatentSync {LOCAL}/LatentSync16")
    venv("latentsync")
    repo = "ByteDance/LatentSync-1.6" if ver == "16" else "ByteDance/LatentSync-1.5"
    if not os.path.exists(f"{d}/checkpoints/latentsync_unet.pt"):
        for fn in ["whisper/tiny.pt", "latentsync_unet.pt"]:
            hf_get("latentsync", repo, fn, "checkpoints", d)
    return d

def ensure_syncnet():
    d = f"{LOCAL}/syncnet_python"
    if not os.path.exists(d): sh(f"git clone -q --depth 1 https://github.com/joonson/syncnet_python {d}")
    venv("syncnet")
    if not os.path.exists(f"{d}/data/syncnet_v2.model"): sh("bash download_model.sh", cwd=d)
    return d
print("workers written:", list(W))

## Test set

| Set | Content | Used for |
|---|---|---|
| FLEURS `en_us` test (CC-BY-4.0) | Read English sentences by many speakers. Matched to FLORES+ `hye_Armn` references by sentence text. Split into 3 speech-rate buckets (words per second), balanced by the gender label. | ASR WER/CER, MT chrF++/COMET and duration fit, TTS voice and accent metrics |
| FLEURS `hy_am` test | Native Armenian recordings | Armenian centroid for the accent metric, "native carrier" prompts, metric anchors |
| HDTF talking-head clips (YouTube; per-clip license varies) | 8-60 s close-up speakers, plus one synthetic two-speaker "dialogue" made by cutting between two clips | End-to-end dub, diarization, lip sync, speed |
| `MY_CLIPS_DIR` | Your TV-pilot clips | End-to-end, same as HDTF |

In [ ]:
# ---- 7. Build the test set -----------------------------------------------------------------
import pyarrow.parquet as pq
from huggingface_hub import hf_hub_download, HfApi
random.seed(SEED); np.random.seed(SEED)
DATA = f"{WORK}/data"
GENDER = {0: "male", 1: "female", 2: "other"}   # FLEURS ClassLabel order (unverified); used only for balancing

def fleurs_table(cfg, split="test"):
    files = [f for f in HfApi().list_repo_files("google/fleurs", repo_type="dataset", revision="refs/convert/parquet")
             if f.startswith(f"{cfg}/{split}/") and f.endswith(".parquet")]
    tabs = [pq.read_table(hf_hub_download("google/fleurs", f, repo_type="dataset", revision="refs/convert/parquet",
                                          local_dir=f"{LOCAL}/fleurs"),
                          columns=["id", "raw_transcription", "transcription", "gender", "audio"]).to_pylist() for f in files]
    return [r for t in tabs for r in t]

def decode16(audio_cell):
    y, sr = sf.read(io.BytesIO(audio_cell["bytes"]), dtype="float32")
    if y.ndim > 1: y = y.mean(1)
    return librosa.resample(y, orig_sr=sr, target_sr=16000) if sr != 16000 else y

def flores(lang):
    rows = {}
    for split in ("dev", "devtest"):
        p = hf_hub_download("openlanguagedata/flores_plus", f"{split}/{lang}.jsonl", repo_type="dataset",
                            token=HF_TOKEN or None, local_dir=f"{LOCAL}/flores")
        for line in open(p):
            r = json.loads(line); rows[(split, r["id"])] = r["text"]
    return rows

MANIFEST = f"{DATA}/manifest.json"
if os.path.exists(MANIFEST):
    M = json.load(open(MANIFEST)); print("manifest loaded")
else:
    eng, hye = flores("eng_Latn"), flores("hye_Armn")
    en2hy = {norm_text(t): hye[k] for k, t in eng.items() if k in hye}
    rows = fleurs_table("en_us")
    cands = []
    for r in rows:
        ref = en2hy.get(norm_text(r["raw_transcription"]))
        if ref: cands.append((r, ref))
    print(f"FLEURS en_us test rows: {len(rows)}  matched to FLORES hye: {len(cands)}")
    seen, items = set(), []
    for r, ref in cands:
        if r["id"] in seen: continue   # one recording per sentence
        y = trim(decode16(r["audio"]), 16000)
        d = len(y) / 16000
        if d < 2.5: continue
        seen.add(r["id"])
        items.append(dict(id=int(r["id"]), text_en=r["raw_transcription"], ref_hy=ref, gender=GENDER.get(r["gender"], "other"),
                          dur=d, wps=len(r["raw_transcription"].split()) / d, _y=y))
    q1, q2 = np.quantile([i["wps"] for i in items], [1/3, 2/3])
    for i in items: i["bucket"] = "slow" if i["wps"] < q1 else ("fast" if i["wps"] > q2 else "medium")
    picked = []
    for b in ("slow", "medium", "fast"):
        pool = [i for i in items if i["bucket"] == b]; random.shuffle(pool)
        for g in ("male", "female"):
            picked += [i for i in pool if i["gender"] == g][: P["n_per_bucket"] // 2]
    os.makedirs(f"{DATA}/fleurs_en", exist_ok=True)
    for k, i in enumerate(picked):
        i["wav16"] = save(f"{DATA}/fleurs_en/{k:03d}.wav", i.pop("_y"), 16000)
    picked_ids = {i["id"] for i in picked}
    rest = [i for i in items if i["id"] not in picked_ids]; random.shuffle(rest)
    en_anchor = [save(f"{DATA}/fleurs_en/anchor_{k:03d}.wav", i["_y"], 16000) for k, i in enumerate(rest[: P["n_centroid"]])]
    hy_rows = fleurs_table("hy_am"); random.shuffle(hy_rows)
    hy_native = []
    os.makedirs(f"{DATA}/fleurs_hy", exist_ok=True)
    for k, r in enumerate(hy_rows[: P["n_centroid"] + 8]):
        y = trim(decode16(r["audio"]), 16000)
        hy_native.append(dict(wav16=save(f"{DATA}/fleurs_hy/{k:03d}.wav", y, 16000),
                              wav24=save(f"{DATA}/fleurs_hy/{k:03d}_24k.wav", librosa.resample(y, orig_sr=16000, target_sr=24000), 24000),
                              text=r["transcription"], raw=r["raw_transcription"], gender=GENDER.get(r["gender"], "other")))
    M = dict(fleurs=[{k: v for k, v in i.items() if not k.startswith("_")} for i in picked], en_anchor=en_anchor, hy_native=hy_native)
    json.dump(M, open(MANIFEST, "w"), ensure_ascii=False, indent=1)
FLEURS = M["fleurs"]
print(pd.DataFrame(FLEURS).groupby(["bucket", "gender"]).size())

In [ ]:
# ---- 8. Video clips: HDTF subset + synthetic dialogue + your clips ---------------------------------------
VID = f"{DATA}/videos"; os.makedirs(VID, exist_ok=True)

def prep_clip(src, dst, start=0.0, length=None):
    t = f"-t {length}" if length else ""
    sh(f'ffmpeg -y -v error -ss {start} -i "{src}" {t} -vf "scale=-2:\'min(720,ih)\',fps=25" '
       f'-c:v libx264 -crf 18 -preset veryfast -c:a aac -ar 44100 -ac 1 "{dst}"')
    return dst

def tsec(s):
    p = [float(x) for x in s.split(":")]; return p[0] * 60 + p[1] if len(p) == 2 else p[0] * 3600 + p[1] * 60 + p[2]

def get_hdtf(n):
    base = "https://raw.githubusercontent.com/MRzzm/HDTF/main/HDTF_dataset"
    urls = dict(l.split()[:2] for l in sh(f"curl -s {base}/RD_video_url.txt").stdout.splitlines() if l.strip())
    times = {}
    for l in sh(f"curl -s {base}/RD_annotion_time.txt").stdout.splitlines():
        parts = l.split()
        if len(parts) >= 2: times[parts[0].replace(".mp4", "")] = parts[1]
    names = [k for k in urls if k in times]; random.Random(SEED).shuffle(names)
    got = []
    for name in names:
        if len(got) >= n: break
        out = f"{VID}/hdtf_{name}.mp4"
        if os.path.exists(out): got.append(out); continue
        a, b = times[name].split("-"); start = tsec(a); length = min(P["max_clip_sec"], tsec(b) - start)
        raw = f"{LOCAL}/raw_{name}.mp4"
        r = sh(f'yt-dlp -q -f "bv*[height<=720]+ba/b[height<=720]" --merge-output-format mp4 '
               f'--download-sections "*{start}-{start + length}" -o "{raw}" "{urls[name]}"', check=False, timeout=600)
        if r.returncode == 0 and os.path.exists(raw):
            got.append(prep_clip(raw, out))
        else:
            print(f"  skip {name}: yt-dlp failed (YouTube often blocks Colab IPs)")
    return got

def make_dialogue(a, b, out):
    """Two speakers alternating with hard cuts: A 6 s, B 6 s, A 4 s (shot changes like a TV scene)."""
    if os.path.exists(out): return out
    f = "scale=1280:720:force_original_aspect_ratio=decrease,pad=1280:720:(ow-iw)/2:(oh-ih)/2,setsar=1,fps=25"
    sh(f'ffmpeg -y -v error -t 6 -i "{a}" -t 6 -i "{b}" -ss 6 -t 4 -i "{a}" -filter_complex '
       f'"[0:v]{f}[v0];[1:v]{f}[v1];[2:v]{f}[v2];[v0][0:a][v1][1:a][v2][2:a]concat=n=3:v=1:a=1[v][a]" '
       f'-map "[v]" -map "[a]" -c:v libx264 -crf 18 -preset veryfast -c:a aac -ar 44100 -ac 1 "{out}"')
    return out

VIDEOS = []
if RUN["video"]:
    hd = get_hdtf(P["n_hdtf"])
    VIDEOS += [dict(name=Path(p).stem, path=p, source="hdtf") for p in hd]
    if len(hd) >= 2:
        VIDEOS.append(dict(name="dialogue_ab", path=make_dialogue(hd[0], hd[1], f"{VID}/dialogue_ab.mp4"), source="synthetic_dialogue"))
    if os.path.isdir(MY_CLIPS_DIR):
        for p in sorted(Path(MY_CLIPS_DIR).glob("*")):
            if p.suffix.lower() in (".mp4", ".mov", ".mkv"):
                out = f"{VID}/mine_{p.stem}.mp4"
                if not os.path.exists(out): prep_clip(str(p), out, 0, P["max_clip_sec"])
                VIDEOS.append(dict(name=f"mine_{p.stem}", path=out, source="mine"))
for v in VIDEOS:
    v["dur"] = float(sh(f'ffprobe -v error -show_entries format=duration -of csv=p=0 "{v["path"]}"').stdout.strip())
print(pd.DataFrame(VIDEOS) if VIDEOS else "No video clips: e2e and lip-sync rows will be skipped.")

## Stage runners
The baseline uses the repo code as it is: `ASRInference`, `TranslationInference`, and `TTSInference` (which falls back to edge-tts), plus `DubbingPipeline._align_and_stitch_segments` and `_process_audio`. This keeps the repo's real bugs in the baseline. Each proposed change swaps exactly one stage.

In [ ]:
# ---- 9. Baseline objects from the repo ------------------------------------------------------------
os.chdir(REPO)
from src.pipeline.pipeline import DubbingPipeline
RP = DubbingPipeline(config_path=f"{REPO}/configs/config.yaml", config_override_path=f"{REPO}/{P['repo_profile']}")
RP.background_separation_enabled = True          # baseline keeps background (repo default intent)
RP.audio_processor.enable_source_separation = True
print("repo baseline: ASR", RP.asr.model_id, "| quant bits", RP.asr.quantize_bits, "| sr", RP.sr,
      "| stretch", RP.min_compress_ratio, RP.max_stretch_ratio, RP.timing_method)

In [ ]:
# ---- 10. ASR: baseline vs WhisperX vs Parakeet, on FLEURS-en --------------------------------------------
from transformers.models.whisper.english_normalizer import BasicTextNormalizer
import jiwer
_bn = BasicTextNormalizer()
def en_norm(t): return re.sub(r"\s+", " ", _bn(t)).strip()

def asr_repo(paths):
    RP.asr.load()
    out = {p: RP.asr.transcribe(p, language="eng") for p in paths}
    RP.asr.free_memory(); free_gpu()
    return {"files": {p: {"text": o.get("text", ""), "segments": o.get("segments", [])} for p, o in out.items()}}

def asr_whisperx(paths):
    venv("whisperx")
    return run_worker("whisperx", "asr_whisperx.py", dict(files=paths, model=P["whisperx_model"],
                      compute_type=P["whisperx_compute"], batch_size=P["whisperx_batch"]))

def asr_parakeet(paths):
    venv("nemo")
    return run_worker("nemo", "asr_nemo.py", dict(files=paths, model="nvidia/parakeet-tdt-0.6b-v2", timestamps=True))

def text_of(sysname, r):
    if sysname == "whisperx": return " ".join(s["text"] for s in r)
    return r["text"]

ASR_TABLE = None
if RUN["asr"]:
    paths = [i["wav16"] for i in FLEURS]; refs = [en_norm(i["text_en"]) for i in FLEURS]; total = sum(i["dur"] for i in FLEURS)
    rows = []
    for name, fn in [("repo_whisper", asr_repo), ("whisperx_turbo", asr_whisperx), ("parakeet_v2", asr_parakeet)]:
        try:
            res, key = cached_stage("asr_fleurs", name, json.dumps(paths), lambda fn=fn: fn(paths), total)
        except Exception as e:
            print(f"{name} failed: {str(e)[-800:]}"); continue
        sysname = "whisperx" if name == "whisperx_turbo" else name
        hyps = [en_norm(text_of(sysname, res["files"][p])) for p in paths]
        perf = STAGE_PERF[key]
        row = dict(system=name, wer=jiwer.wer(refs, hyps), cer=jiwer.cer(refs, hyps),
                   rtf=perf["seconds"] / total, vram_peak_gb=max(perf.get("torch_peak_gb", 0), perf.get("smi_peak_gb", 0)))
        for b in ("slow", "medium", "fast"):
            idx = [k for k, i in enumerate(FLEURS) if i["bucket"] == b]
            if idx: row[f"wer_{b}"] = jiwer.wer([refs[k] for k in idx], [hyps[k] for k in idx])
        rows.append(row)
    ASR_TABLE = pd.DataFrame(rows); display(ASR_TABLE)

In [ ]:
# ---- 11. MT systems (baseline SeamlessM4T, TranslateGemma, MADLAD, duration-aware, Gemini) ------------------------
from transformers import AutoProcessor, AutoTokenizer, AutoModelForImageTextToText, AutoModelForSeq2SeqLM, BitsAndBytesConfig

def quant_kwargs():
    compute = torch.bfloat16 if BF16 else torch.float32
    kw = dict(device_map="auto", token=HF_TOKEN or None, torch_dtype=compute,
              max_memory={0: f"{int(VRAM_GB * 0.85)}GiB", "cpu": "48GiB"})   # spill to CPU instead of OOM
    if P["llm_quant"] == "nf4":
        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                       bnb_4bit_compute_dtype=compute, bnb_4bit_use_double_quant=True)
    elif P["llm_quant"] == "int8":
        kw["quantization_config"] = BitsAndBytesConfig(load_in_8bit=True)
    return kw

class ChatLLM:
    """Gemma-3 / TranslateGemma via the multimodal chat template (text only)."""
    def __init__(self, mid):
        self.proc = AutoProcessor.from_pretrained(mid, token=HF_TOKEN or None)
        self.m = AutoModelForImageTextToText.from_pretrained(mid, **quant_kwargs()).eval()
    def _gen(self, msgs, **gen):
        inp = self.proc.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_dict=True,
                                            return_tensors="pt").to(self.m.device)
        with torch.inference_mode():
            out = self.m.generate(**inp, **gen)
        n = inp["input_ids"].shape[-1]
        return [self.proc.decode(o[n:], skip_special_tokens=True).strip() for o in out]
    def translate_nbest(self, text, k=4):
        for code in ("hy", "hy-AM"):
            try:
                msgs = [{"role": "user", "content": [{"type": "text", "source_lang_code": "en", "target_lang_code": code, "text": text}]}]
                return self._gen(msgs, max_new_tokens=256, num_beams=k, num_return_sequences=k, do_sample=False)
            except Exception as e:
                last = e
        raise last
    def chat(self, prompt, temperature=0.7):
        msgs = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
        return self._gen(msgs, max_new_tokens=400, do_sample=True, temperature=temperature, top_p=0.95)[0]
    def close(self):
        del self.m; free_gpu()

def gemini(prompt):
    from google import genai
    return genai.Client(api_key=GEMINI_API_KEY).models.generate_content(model=GEMINI_MODEL, contents=prompt).text.strip()

GEMINI_TRANSLATE = ("Translate the English line into natural spoken Eastern Armenian (Republic of Armenia orthography) "
                    "for a TV dub. Keep register and names. Output only the translation.\n"
                    "Previous line: {prev}\nLine: {src}\nNext line: {next}\nSpeaker gender: {gender}")
REWRITE = """You are a dubbing translator into Eastern Armenian (Republic of Armenia standard orthography).
English line: {src}
Previous line: {prev}
Next line: {next}
Speaker gender: {gender}. Keep the register (formal/informal "you") consistent with the context.
Draft translations: {drafts}
Write 4 different Eastern Armenian translations that keep the meaning, sound natural when spoken,
and have about {budget} syllables each (allowed {lo}-{hi}). Shorten by dropping fillers and repetition, never by dropping facts or names.
Example of strong shortening: "I really don't think that's a good idea right now." -> "Հիմա դա լավ միտք չէ։"
Output only a JSON list of 4 strings."""

def parse_list(s):
    m = re.search(r"\[.*\]", s, re.S)
    if m:
        try: return [str(x).strip() for x in json.loads(m.group(0)) if str(x).strip()]
        except Exception: pass
    return [l.strip(" -*0123456789.\"") for l in s.splitlines() if hy_ratio(l) > 0.6]

def fit_err(text, slot, rate): return abs(hy_syllables(text) / rate - slot) / max(slot, 0.3)

def pick(cands, slot, rate):
    ok = [c for c in cands if c and hy_ratio(c) >= 0.6]
    if not ok: return (cands[0] if cands else ""), 9.9
    best = min(ok, key=lambda c: fit_err(c, slot, rate)); return best, fit_err(best, slot, rate)

def mt_run(system, lines, rate, tol=0.10):
    """lines: [{src, slot, prev, next, gender}] -> list of Armenian strings. Loads models once."""
    out = [""] * len(lines)
    if system == "seamless":
        RP.translator.load()
        for k, l in enumerate(lines):
            out[k] = RP.translator.translate(l["src"], "eng", "hye")["tgt_text"]   # repo: returns English on error
        RP.translator.free_memory(); free_gpu(); return out
    if system == "madlad":
        tok = AutoTokenizer.from_pretrained(P["madlad"]); m = AutoModelForSeq2SeqLM.from_pretrained(P["madlad"], **quant_kwargs()).eval()
        for k, l in enumerate(lines):
            ids = tok("<2hy> " + l["src"], return_tensors="pt").input_ids.to(m.device)
            with torch.inference_mode():
                out[k] = tok.decode(m.generate(input_ids=ids, max_new_tokens=256, num_beams=4)[0], skip_special_tokens=True)
        del m; free_gpu(); return out
    if system.startswith("gemini"):
        cands = [[gemini(GEMINI_TRANSLATE.format(**l))] for l in lines]
    else:   # tgemma, tgemma_dur
        llm = ChatLLM(P["tgemma"])
        cands = [llm.translate_nbest(l["src"], k=4 if system.endswith("_dur") else 1) for l in lines]
        llm.close()
    if not system.endswith("_dur"):
        return [c[0] for c in cands]
    need = [k for k, l in enumerate(lines) if pick(cands[k], l["slot"], rate)[1] > tol]
    rewriter = None if system.startswith("gemini") else ChatLLM(P["rewrite_llm"])
    for k in need:
        l = lines[k]; budget = max(2, round(l["slot"] * rate))
        prompt = REWRITE.format(src=l["src"], prev=l["prev"], next=l["next"], gender=l["gender"], drafts=json.dumps(cands[k][:2], ensure_ascii=False),
                                budget=budget, lo=round(budget * 0.9), hi=round(budget * 1.1))
        try:
            cands[k] = cands[k] + parse_list(gemini(prompt) if rewriter is None else rewriter.chat(prompt))
        except Exception as e:
            print("rewrite failed:", str(e)[:200])
    if rewriter: rewriter.close()
    return [pick(c, l["slot"], rate)[0] for c, l in zip(cands, lines)]

MT_SYSTEMS = ["seamless", "tgemma", "madlad", "tgemma_dur"] + (["gemini", "gemini_dur"] if RUN["api_rows"] and GEMINI_API_KEY else [])
MT_COMMERCIAL = dict(seamless=False, tgemma=True, madlad=True, tgemma_dur=True, gemini=True, gemini_dur=True)
print("MT systems:", MT_SYSTEMS)

In [ ]:
# ---- 12. TTS + voice-conversion systems ------------------------------------------------------------
import shlex
EDGE_VOICE = {"female": "hy-AM-AnahitNeural", "male": "hy-AM-HaykNeural", "other": "hy-AM-AnahitNeural"}

def edge_say(text, voice, out24):
    mp3 = out24.replace(".wav", ".mp3")
    sh(f'edge-tts --voice {voice} --text {shlex.quote(text)} --write-media "{mp3}"')
    sh(f'ffmpeg -y -v error -i "{mp3}" -ar 24000 -ac 1 "{out24}"'); os.remove(mp3); return out24

def azure_say(text, voice, out24, rate_pct=0):
    import azure.cognitiveservices.speech as speechsdk
    from xml.sax.saxutils import escape
    cfg = speechsdk.SpeechConfig(subscription=AZURE_SPEECH_KEY, region=AZURE_REGION)
    cfg.set_speech_synthesis_output_format(speechsdk.SpeechSynthesisOutputFormat.Riff24Khz16BitMonoPcm)
    ssml = (f'<speak version="1.0" xmlns="http://www.w3.org/2001/10/synthesis" xml:lang="hy-AM"><voice name="{voice}">'
            f'<prosody rate="{rate_pct:+d}%">{escape(text)}</prosody></voice></speak>')
    r = speechsdk.SpeechSynthesizer(speech_config=cfg, audio_config=None).speak_ssml_async(ssml).get()
    open(out24, "wb").write(r.audio_data); return out24

def fish_say(text, ref_wav, ref_text, out):
    from fish_audio_sdk import Session, TTSRequest, ReferenceAudio   # SDK API shape unverified
    s = Session(FISH_API_KEY)
    req = TTSRequest(text=text, format="wav", references=[ReferenceAudio(audio=open(ref_wav, "rb").read(), text=ref_text)])
    try: chunks = s.tts(req, backend=FISH_MODEL)
    except TypeError: chunks = s.tts(req)
    with open(out, "wb") as f:
        for c in chunks: f.write(c)
    return out

def repo_say(text, out):
    RP.tts.load()
    r = RP.tts.synthesize(text=text, reference_audio_path=None, emotion="neutral", language="hy")
    return save(out, r["audio"], r["sample_rate"])

def openvoice_batch(items):
    """items: [{src, ref, out}] -> converts timbre of src to ref (language-agnostic)."""
    venv("openvoice")
    return run_worker("openvoice", "vc_openvoice.py", dict(items=items))

def omnivoice_batch(items):
    venv("omnivoice")
    return run_worker("omnivoice", "tts_omnivoice.py", dict(items=items, num_step=P["omni_steps"]))

def native_carrier(gender):
    """A native Armenian FLEURS prompt of the same gender (accent comes from here, timbre from OpenVoice later)."""
    pool = [h for h in M["hy_native"][P["n_centroid"]:] if h["gender"] == gender] or M["hy_native"][P["n_centroid"]:]
    return pool[0]

def tts_run(system, jobs, outdir):
    """jobs: [{text, ref24, ref_text, gender, slot}] -> list of wav paths (or None). One model load per call."""
    os.makedirs(outdir, exist_ok=True)
    outs = [f"{outdir}/{k:04d}.wav" for k in range(len(jobs))]
    if system == "repo_edge":
        return [repo_say(j["text"], o) if j["text"].strip() else None for j, o in zip(jobs, outs)]
    if system in ("edge_ov", "azure_ov"):
        raw = []
        for j, o in zip(jobs, outs):
            if not j["text"].strip(): raw.append(None); continue
            f = o.replace(".wav", "_base.wav")
            raw.append(azure_say(j["text"], EDGE_VOICE[j["gender"]], f) if system == "azure_ov" else edge_say(j["text"], EDGE_VOICE[j["gender"]], f))
        items = [dict(src=r, ref=j["ref24"], out=o) for r, j, o in zip(raw, jobs, outs) if r]
        res = openvoice_batch(items)
        ok = {i["out"] for i in res["items"] if i["ok"]}
        return [o if o in ok else None for o in outs]
    if system.startswith("omni"):
        items = []
        for j, o in zip(jobs, outs):
            if not j["text"].strip(): continue
            it = dict(text=j["text"], ref_audio=j["ref24"], ref_text=j["ref_text"], out=o)
            if system in ("omni_dur", "omni_native_ov"): it["duration"] = round(max(0.6, j["slot"]), 2)
            if system == "omni_native_ov":
                c = native_carrier(j["gender"]); it.update(ref_audio=c["wav24"], ref_text=c["raw"], out=o.replace(".wav", "_carrier.wav"))
            items.append(it)
        res = omnivoice_batch(items)
        ok = {i["out"] for i in res["items"] if i["ok"]}
        if system != "omni_native_ov":
            return [o if o in ok else None for o in outs]
        vc = [dict(src=o.replace(".wav", "_carrier.wav"), ref=j["ref24"], out=o) for j, o in zip(jobs, outs) if o.replace(".wav", "_carrier.wav") in ok]
        ok2 = {i["out"] for i in openvoice_batch(vc)["items"] if i["ok"]}
        return [o if o in ok2 else None for o in outs]
    if system == "fish":
        return [fish_say(j["text"], j["ref24"], j["ref_text"], o) if j["text"].strip() else None for j, o in zip(jobs, outs)]
    raise ValueError(system)

TTS_SYSTEMS = ["repo_edge", "edge_ov", "omni", "omni_dur", "omni_native_ov"]
if RUN["api_rows"] and AZURE_SPEECH_KEY: TTS_SYSTEMS.append("azure_ov")
if RUN["api_rows"] and FISH_API_KEY: TTS_SYSTEMS.append("fish")
TTS_COMMERCIAL = dict(repo_edge=False, edge_ov="proxy", azure_ov=True, omni=False, omni_dur=False, omni_native_ov=False, fish=True)
print("TTS systems:", TTS_SYSTEMS)

## Metrics
- **ASR:** WER and CER after Whisper's basic text normalizer.
- **MT:** chrF++ (sacrebleu, `word_order=2`) and COMET-22 (`Unbabel/wmt22-comet-da`, reference-based) against FLORES+ `hye_Armn`. Duration fit = estimated Armenian speech time (syllables / calibrated TTS rate) vs the source slot.
- **Voice:** speaker similarity (SECS) as cosine similarity of WavLM-base-plus-SV and ECAPA (VoxCeleb) embeddings against the source speaker. Naturalness: UTMOSv2 as a MOS predictor (trained on English and Japanese, so it is a weak proxy for Armenian).
- **Intelligibility:** back-ASR CER with `nvidia/stt_hy_fastconformer_hybrid_large_pc` against the text that was synthesized.
- **Accent:** see the method cell below.
- **Lip sync:** LSE-C and LSE-D from SyncNet v2 (`syncnet_python`). Visual quality: FID on mouth crops (source vs dub) and identity CSIM (FaceNet).
- **Speed:** RTF = stage seconds / audio seconds. VRAM peak is the max of torch (in-process) and nvidia-smi (whole device, includes workers).

In [ ]:
# ---- 13. Metric models (lazy; unloaded before heavy stages) ------------------------------------------
from scipy.stats import pearsonr
class VoiceMetrics:
    def __init__(self): self.ok = False
    def load(self):
        if self.ok: return self
        from transformers import AutoFeatureExtractor, WavLMForXVector
        try:
            from speechbrain.inference.speaker import EncoderClassifier as SpkEnc
            from speechbrain.inference.classifiers import EncoderClassifier as LidEnc
        except ImportError:
            from speechbrain.pretrained import EncoderClassifier as SpkEnc
            LidEnc = SpkEnc
        self.fe = AutoFeatureExtractor.from_pretrained("microsoft/wavlm-base-plus-sv")
        self.wavlm = WavLMForXVector.from_pretrained("microsoft/wavlm-base-plus-sv").cuda().eval()
        self.ecapa = SpkEnc.from_hparams("speechbrain/spkrec-ecapa-voxceleb", savedir=f"{LOCAL}/sb_ecapa", run_opts={"device": "cuda"})
        self.lid = LidEnc.from_hparams("speechbrain/lang-id-voxlingua107-ecapa", savedir=f"{LOCAL}/sb_lid", run_opts={"device": "cuda"})
        lab = self.lid.hparams.label_encoder.lab2ind
        self.hy_idx = next(v for k, v in lab.items() if str(k).split(":")[0].strip() == "hy")
        try:
            import utmosv2; self.utmos = utmosv2.create_model(pretrained=True)
        except Exception as e:
            print("UTMOSv2 unavailable:", str(e)[:200]); self.utmos = None
        self.ok = True; return self
    def unload(self):
        if self.ok:
            for a in ("wavlm", "ecapa", "lid", "utmos"): setattr(self, a, None)
            self.ok = False; free_gpu()
    @torch.inference_mode()
    def spk(self, y16):
        inp = self.fe([y16], sampling_rate=16000, return_tensors="pt", padding=True).to("cuda")
        a = torch.nn.functional.normalize(self.wavlm(**inp).embeddings, dim=-1)[0].cpu()
        b = torch.nn.functional.normalize(self.ecapa.encode_batch(torch.tensor(y16)[None].cuda()).squeeze(), dim=-1).cpu()
        return a, b
    def secs(self, y16, ref16):
        (a1, b1), (a2, b2) = self.spk(y16), self.spk(ref16)
        return float(a1 @ a2), float(b1 @ b2)
    @torch.inference_mode()
    def lang(self, y16):
        sig = torch.tensor(y16)[None].cuda()
        logp = self.lid.classify_batch(sig)[0][0]
        emb = torch.nn.functional.normalize(self.lid.encode_batch(sig).squeeze(), dim=-1).cpu()
        return float(torch.exp(logp[self.hy_idx])), emb
    def mos(self, path):
        if self.utmos is None: return float("nan")
        try: return float(self.utmos.predict(input_path=path))
        except Exception: return float("nan")
VM = VoiceMetrics()

def hy_norm(t): return re.sub(r"\s+", " ", re.sub(r"[^\w\s]", " ", unicodedata.normalize("NFKC", t).lower())).strip()

def back_asr(paths):
    """Armenian back-transcription with the NVIDIA FastConformer (CC-BY-4.0)."""
    venv("nemo")
    tmp = []
    for p in paths:
        q = f"{LOCAL}/bk_{_h(p)}.wav"; save(q, load(p, 16000), 16000); tmp.append(q)
    r = run_worker("nemo", "asr_nemo.py", dict(files=tmp, model="nvidia/stt_hy_fastconformer_hybrid_large_pc"))
    return [r["files"][q]["text"] for q in tmp]

def prosody(y16):
    f0, _, _ = librosa.pyin(y16, fmin=65, fmax=500, sr=16000, frame_length=1024)
    f0 = f0[~np.isnan(f0)]
    st = 12 * np.log2(f0 / 100.0) if len(f0) else np.array([np.nan])
    db = 20 * np.log10(librosa.feature.rms(y=y16)[0] + 1e-6); sp = db[db > db.max() - 35]
    return dict(f0_mean=float(np.nanmean(st)), f0_std=float(np.nanstd(st)),
                e_range=float(np.percentile(sp, 95) - np.percentile(sp, 10)) if len(sp) > 5 else np.nan)

_CENT = {}
def centroids():
    """LID-embedding centroids of real English and real Armenian speech (FLEURS, other speakers)."""
    if not _CENT:
        VM.load()
        en = torch.stack([VM.lang(load(p))[1] for p in M["en_anchor"][:-5]])   # last 5 are held out as anchors
        hy = torch.stack([VM.lang(load(h["wav16"]))[1] for h in M["hy_native"][: P["n_centroid"]]])
        _CENT["en"] = torch.nn.functional.normalize(en.mean(0), dim=-1); _CENT["hy"] = torch.nn.functional.normalize(hy.mean(0), dim=-1)
    return _CENT

def accent_leak(emb):
    """ALI = cos(dub, English centroid) - cos(dub, Armenian centroid). Lower = more native-Armenian-like."""
    c = centroids(); return float(emb @ c["en"] - emb @ c["hy"])

def voice_eval(items):
    """items: [{wav, text_hy, ref16_path, src16_path (for style), src_text_en}] -> per-item metrics (+ back-ASR CER)."""
    VM.load(); rows = []
    for it in items:
        if not it.get("wav") or not os.path.exists(it["wav"]): rows.append(dict(failed=1)); continue
        y = trim(load(it["wav"]), 16000)
        if len(y) < 1600: rows.append(dict(failed=1)); continue
        s_wavlm, s_ecapa = VM.secs(y, load(it["ref16_path"]))
        p_hy, emb = VM.lang(y)
        src = trim(load(it["src16_path"]), 16000)
        pd_, ps = prosody(y), prosody(src)
        rows.append(dict(failed=0, secs_wavlm=s_wavlm, secs_ecapa=s_ecapa, utmos=VM.mos(it["wav"]), p_hy=p_hy, ali=accent_leak(emb),
                         d_f0_st=abs(pd_["f0_mean"] - ps["f0_mean"]), f0_std_ratio=pd_["f0_std"] / max(ps["f0_std"], 1e-3),
                         e_range_ratio=pd_["e_range"] / max(ps["e_range"], 1e-3),
                         dub_rate=hy_syllables(it["text_hy"]) / max(len(y) / 16000, 0.3),
                         src_rate=en_syllables(it.get("src_text_en", "")) / max(len(src) / 16000, 0.3)))
    VM.unload()
    ok = [k for k, r in enumerate(rows) if not r["failed"]]
    hyps = back_asr([items[k]["wav"] for k in ok]) if ok else []
    for k, h in zip(ok, hyps):
        rows[k]["cer_hy"] = jiwer.cer(hy_norm(items[k]["text_hy"]) or "-", hy_norm(h) or "-")
    return pd.DataFrame(rows)

def summarize_voice(df):
    d = df[df.failed == 0] if "failed" in df else df
    out = {c: d[c].mean() for c in ["secs_wavlm", "secs_ecapa", "utmos", "cer_hy", "p_hy", "ali", "d_f0_st", "f0_std_ratio", "e_range_ratio"] if c in d}
    if len(d) >= 5 and "src_rate" in d and d.src_rate.std() > 0 and d.dub_rate.std() > 0:
        out["rate_corr"] = pearsonr(d.src_rate, d.dub_rate)[0]
    out["fail_rate"] = float(df.failed.mean()) if "failed" in df else 0.0
    return out
print("metric functions ready")

### Accent and style test: method and limits
**Goal for a default dub:** native Eastern Armenian pronunciation, with the actor's style kept (pitch level and range, energy dynamics, tempo changes). For characters who are meant to sound foreign, the target flips, and the same numbers show it.

**Method (automatic):**
1. **Nativeness.** VoxLingua107 posterior `p_hy` plus Armenian back-ASR CER. Native speech gives high `p_hy` and low CER.
2. **Accent-leakage index (ALI).** In the VoxLingua107 embedding space, `ALI = cos(dub, English centroid) - cos(dub, Armenian centroid)`. Centroids come from real FLEURS speakers who are not in the test set. The anchor rows "real hy" and "real en" give the scale: a native dub should sit near "real hy".
3. **Style.** Per utterance: `|delta F0 mean|` in semitones (dub vs source), F0 std ratio (pitch range), energy-range ratio, and `rate_corr` = Pearson correlation between source and dub speaking rate across lines. The speed buckets (slow/medium/fast) make the last one meaningful.
4. **Human check.** ABX sheet (exported below): native Eastern Armenian listeners judge "Which sample sounds like a native speaker?" and "Which matches the actor's delivery?".

**Limits:**
- LID models confuse channel and recording conditions with language and accent. TTS audio is cleaner than FLEURS, which can shift `p_hy` and ALI regardless of accent. Compare systems with each other, not with an absolute threshold.
- ALI measures "Armenian-likeness" in a language-ID space, not phonetic accent. It is not validated against human accent ratings for Armenian (unverified).
- Back-ASR CER mixes intelligibility, text normalization, and ASR errors. The FastConformer was trained mostly on read speech.
- Cross-language prosody is not directly comparable (different stress and syllable structure). Only utterance-level statistics are used, never frame alignment.
- Gender split by the FLEURS label, and pitch-based gender for video speakers, are coarse.
- SECS from a cross-lingual prompt is biased upward when the prompt and the SECS target are the same recording (the FLEURS component test). On video, held-out turns are used where they exist.

In [ ]:
# ---- 14. Component tests on FLEURS: MT then TTS --------------------------------------------------------
from sacrebleu.metrics import CHRF
def comet_score(src, hyp, ref):
    from comet import download_model, load_from_checkpoint
    m = load_from_checkpoint(download_model("Unbabel/wmt22-comet-da"))
    out = m.predict([dict(src=s, mt=h, ref=r) for s, h, r in zip(src, hyp, ref)], batch_size=16, gpus=1, progress_bar=False)
    del m; free_gpu(); return float(out.system_score)

# speaking-rate calibration (Armenian syllables / second) from edge-tts on FLORES refs; used for MT duration budgets
cal = FLEURS[:6]
calp = [edge_say(i["ref_hy"], "hy-AM-AnahitNeural", f"{LOCAL}/cal_{k}.wav") for k, i in enumerate(cal)]
TTS_RATE = float(np.median([hy_syllables(i["ref_hy"]) / (len(trim(load(p), 16000)) / 16000) for i, p in zip(cal, calp)]))
print(f"calibrated Armenian TTS rate: {TTS_RATE:.2f} syllables/s")

FL_LINES = [dict(src=i["text_en"], slot=i["dur"], prev="", next="", gender=i["gender"]) for i in FLEURS]
MT_OUT, MT_TABLE = {}, None
if RUN["mt"]:
    VM.unload(); rows = []
    srcs, refs = [i["text_en"] for i in FLEURS], [i["ref_hy"] for i in FLEURS]
    for s in MT_SYSTEMS:
        try:
            hyp, key = cached_stage("mt_fleurs", s, json.dumps(srcs) + f"{TTS_RATE:.2f}", lambda s=s: mt_run(s, FL_LINES, TTS_RATE))
        except Exception as e:
            print(f"{s} failed: {str(e)[-800:]}"); continue
        MT_OUT[s] = hyp
        ratio = np.array([hy_syllables(h) / TTS_RATE / l["slot"] for h, l in zip(hyp, FL_LINES)])
        rows.append(dict(system=s, commercial_ok=MT_COMMERCIAL[s], chrfpp=CHRF(word_order=2).corpus_score(hyp, [refs]).score,
                         comet22=comet_score(srcs, hyp, refs), english_leak=float(np.mean([hy_ratio(h) < 0.6 for h in hyp])),
                         fit_outside_15=float(np.mean(np.abs(ratio - 1) > 0.15)), fit_ratio_median=float(np.median(ratio)),
                         rtf=STAGE_PERF[key]["seconds"] / sum(l["slot"] for l in FL_LINES),
                         vram_peak_gb=max(STAGE_PERF[key].get("torch_peak_gb", 0), STAGE_PERF[key].get("smi_peak_gb", 0))))
    MT_TABLE = pd.DataFrame(rows); display(MT_TABLE)

TTS_TABLE = None
if RUN["tts"]:
    rows = []
    jobs = []
    for k, i in enumerate(FLEURS):
        ref24 = save(f"{DATA}/fleurs_en/{k:03d}_24k.wav", librosa.resample(load(i["wav16"]), orig_sr=16000, target_sr=24000), 24000)
        jobs.append(dict(text=i["ref_hy"], ref24=ref24, ref_text=i["text_en"], gender=i["gender"], slot=i["dur"]))
    for s in TTS_SYSTEMS:
        try:
            outs, key = cached_stage("tts_fleurs", s, json.dumps([j["text"] for j in jobs]),
                                     lambda s=s: tts_run(s, jobs, f"{WORK}/out/tts_fleurs/{s}"), sum(i["dur"] for i in FLEURS))
        except Exception as e:
            print(f"{s} failed: {str(e)[-800:]}"); continue
        df = voice_eval([dict(wav=o, text_hy=i["ref_hy"], ref16_path=i["wav16"], src16_path=i["wav16"], src_text_en=i["text_en"])
                         for o, i in zip(outs, FLEURS)])
        fitr = [dur(o) / i["dur"] for o, i in zip(outs, FLEURS) if o and os.path.exists(o)]
        perf = STAGE_PERF[key]
        rows.append(dict(system=s, commercial_ok=TTS_COMMERCIAL[s], **summarize_voice(df),
                         dur_ratio_median=float(np.median(fitr)) if fitr else np.nan,
                         rtf=perf["seconds"] / sum(i["dur"] for i in FLEURS), vram_peak_gb=max(perf.get("torch_peak_gb", 0), perf.get("smi_peak_gb", 0))))
    # anchors: real Armenian and real English speech (scale for p_hy / ALI / UTMOS / CER)
    hold = M["hy_native"][P["n_centroid"]: P["n_centroid"] + 5]
    adf = voice_eval([dict(wav=h["wav16"], text_hy=h["raw"], ref16_path=h["wav16"], src16_path=h["wav16"]) for h in hold])
    rows.append(dict(system="ANCHOR real hy (FLEURS)", **summarize_voice(adf)))
    en_hold = M["en_anchor"][-5:]
    VM.load(); en_ali = [accent_leak(VM.lang(trim(load(p), 16000))[1]) for p in en_hold]; en_phy = [VM.lang(trim(load(p), 16000))[0] for p in en_hold]; VM.unload()
    rows.append(dict(system="ANCHOR real en (FLEURS)", ali=float(np.mean(en_ali)), p_hy=float(np.mean(en_phy))))
    TTS_TABLE = pd.DataFrame(rows); display(TTS_TABLE)

In [ ]:
# ---- 15. End-to-end video pipeline pieces -----------------------------------------------------------
from demucs.pretrained import get_model as demucs_model
from demucs.apply import apply_model
import pyloudnorm as pyln

def separate(clip):
    """Demucs htdemucs on GPU, stereo in; returns paths of vocals / accompaniment at 44.1 kHz mono."""
    d = f"{WORK}/out/sep/{clip['name']}"; v, a = f"{d}/vocals.wav", f"{d}/accomp.wav"
    if os.path.exists(a): return v, a
    os.makedirs(d, exist_ok=True)
    wav = f"{d}/orig.wav"; sh(f'ffmpeg -y -v error -i "{clip["path"]}" -ar 44100 -ac 2 "{wav}"')
    y, sr = sf.read(wav, dtype="float32"); m = demucs_model("htdemucs").cuda().eval()
    with torch.inference_mode():
        src = apply_model(m, torch.tensor(y.T)[None].cuda(), device="cuda", split=True, overlap=0.25)[0].cpu().numpy()
    names = m.sources; vi = names.index("vocals")
    save(v, src[vi].mean(0), sr); save(a, sum(src[i] for i in range(len(names)) if i != vi).mean(0), sr)
    del m; free_gpu(); return v, a

def assign_speakers(words, turns):
    for w in words:
        if w.get("start") is None: continue
        best, bo = None, 0.0
        for s, e, spk in turns:
            o = min(e, w["end"]) - max(s, w["start"])
            if o > bo: best, bo = spk, o
        if best is None and turns:
            best = min(turns, key=lambda t: min(abs(t[0] - w["start"]), abs(t[1] - w["end"])))[2]
        w["spk"] = best or "S0"
    return words

def merge_lines(words, gap=0.6, max_len=8.0):
    lines, cur = [], None
    for w in words:
        if w.get("start") is None:
            if cur: cur["text"] += " " + w["word"]
            continue
        spk = w.get("spk", "S0")
        new = (cur is None or spk != cur["speaker"] or w["start"] - cur["end"] > gap or cur["end"] - cur["start"] > max_len
               or (re.search(r"[.?!]$", cur["text"]) and w["start"] - cur["end"] > 0.25))
        if new:
            cur = dict(start=w["start"], end=w["end"], text=w["word"], speaker=spk); lines.append(cur)
        else:
            cur["end"] = w["end"]; cur["text"] += " " + w["word"]
    return lines

def finish_lines(lines, total):
    for k, l in enumerate(lines):
        l["prev"] = lines[k - 1]["text"] if k else ""; l["next"] = lines[k + 1]["text"] if k + 1 < len(lines) else ""
        l["slot"] = max(0.3, min(l["end"], total) - l["start"])
    return lines

def asr_stage(clips, cfg):
    """-> {clip: lines}. repo = Whisper chunks (no words, single speaker); whisperx = sentence lines (+ diarization)."""
    def run():
        out = {}
        wavs = {}
        for c in clips:
            w = f"{WORK}/out/sep/{c['name']}/orig16.wav"
            if not os.path.exists(w): os.makedirs(Path(w).parent, exist_ok=True); sh(f'ffmpeg -y -v error -i "{c["path"]}" -ar 16000 -ac 1 "{w}"')
            wavs[c["name"]] = w
        if cfg["asr"] == "repo":
            r = asr_repo(list(wavs.values()))["files"]
            for c in clips:
                segs = r[wavs[c["name"]]]["segments"]
                out[c["name"]] = finish_lines([dict(start=s["start"], end=s["end"] or c["dur"], text=s["text"], speaker="S0") for s in segs if s["text"].strip()], c["dur"])
            return out
        r = asr_whisperx(list(wavs.values()))["files"]
        turns = {}
        if cfg["diar"]:
            venv("pyannote")
            turns = run_worker("pyannote", "diar_pyannote.py", dict(files=list(wavs.values()), model="pyannote/speaker-diarization-community-1", hf_token=HF_TOKEN))["files"]
        for c in clips:
            words = [w for s in r[wavs[c["name"]]] for w in s["words"]]
            if cfg["diar"]: words = assign_speakers(words, turns.get(wavs[c["name"]], []))
            out[c["name"]] = finish_lines(merge_lines(words), c["dur"])
        return out
    key = json.dumps({k: cfg[k] for k in ("asr", "diar")}, sort_keys=True) + json.dumps([c["name"] for c in clips])
    return cached_stage("e2e_asr", cfg["asr"] + ("_diar" if cfg["diar"] else ""), key, run, sum(c["dur"] for c in clips))

def speaker_refs(clip, lines, tag, min_s=3.0, max_s=10.0):
    """Per speaker: prompt clip (3-10 s of its longest lines, from the vocal stem) + held-out clip for SECS."""
    vpath, _ = separate(clip); y = load(vpath, 24000); refs = {}
    d = f"{WORK}/out/refs/{tag}/{clip['name']}"; os.makedirs(d, exist_ok=True)
    for spk in sorted({l["speaker"] for l in lines}):
        ls = sorted([l for l in lines if l["speaker"] == spk], key=lambda l: l["end"] - l["start"], reverse=True)
        use, tot = [], 0.0
        for l in ls:
            if tot >= max_s: break
            use.append(l); tot += l["end"] - l["start"]
        held = [l for l in ls if l not in use] or use
        cat = lambda L, sr: np.concatenate([np.concatenate([y[int(l["start"] * 24000): int(min(l["end"], l["start"] + max_s) * 24000)], np.zeros(3600, np.float32)]) for l in L])
        pr = cat(use, 24000)[: int(max_s * 24000)]
        p24 = save(f"{d}/{spk}_prompt24.wav", pr, 24000)
        h16 = save(f"{d}/{spk}_held16.wav", librosa.resample(cat(held, 24000), orig_sr=24000, target_sr=16000), 16000)
        f0 = librosa.pyin(librosa.resample(pr, orig_sr=24000, target_sr=16000), fmin=65, fmax=400, sr=16000)[0]
        gender = "female" if np.nanmedian(f0) > 165 else "male"
        refs[spk] = dict(ref24=p24, held16=h16, ref_text=" ".join(l["text"] for l in use), gender=gender, heldout=len(held) > 0 and held != use)
    return refs

def mt_stage(clips, lines_by, refs_by, cfg):
    def run():
        flat = [(c["name"], k, l) for c in clips for k, l in enumerate(lines_by[c["name"]])]
        L = [dict(src=l["text"], slot=l["slot"], prev=l["prev"], next=l["next"], gender=refs_by[n][l["speaker"]]["gender"]) for n, _, l in flat]
        VM.unload(); hyp = mt_run(cfg["mt"], L, TTS_RATE)
        out = {c["name"]: [""] * len(lines_by[c["name"]]) for c in clips}
        for (n, k, _), h in zip(flat, hyp): out[n][k] = h
        return out
    key = json.dumps({k: cfg[k] for k in ("asr", "diar", "mt")}, sort_keys=True) + json.dumps([c["name"] for c in clips])
    return cached_stage("e2e_mt", cfg["mt"], key, run)

def tts_stage(clips, lines_by, texts_by, refs_by, cfg):
    def run():
        out = {}
        for c in clips:
            n = c["name"]
            jobs = [dict(text=t, slot=l["slot"], **{k: refs_by[n][l["speaker"]][k] for k in ("ref24", "ref_text", "gender")})
                    for l, t in zip(lines_by[n], texts_by[n])]
            out[n] = tts_run(cfg["tts"], jobs, f"{WORK}/out/e2e_tts/{_h(key)}/{n}")
        return out
    key = json.dumps({k: cfg[k] for k in ("asr", "diar", "mt", "tts")}, sort_keys=True) + json.dumps([c["name"] for c in clips])
    return cached_stage("e2e_tts", cfg["tts"], key, run, sum(c["dur"] for c in clips))

def rb_stretch(y, sr, ratio):
    """Rubberband time-stretch: ratio = output/input duration."""
    i, o = tempfile.mktemp(suffix=".wav", dir=LOCAL), tempfile.mktemp(suffix=".wav", dir=LOCAL)
    save(i, y, sr); sh(f"rubberband -q {RB_ENGINE} -t {ratio:.4f} {i} {o}"); z, _ = sf.read(o, dtype="float32"); return z

def fit_fixed(lines, wavs, total, sr=24000, min_ratio=0.82, max_slow=1.12, borrow=0.6, early=0.12):
    """Respect clamps, borrow following silence, never overwrite the next line, 15 ms fades, overlap-add."""
    out = np.zeros(int(total * sr) + sr, np.float32); info, prev_end = [], 0.0
    for k, (l, w) in enumerate(zip(lines, wavs)):
        if not w or not os.path.exists(w): info.append(None); continue
        a = trim(load(w, sr), sr); d = len(a) / sr
        nxt = lines[k + 1]["start"] if k + 1 < len(lines) else total
        start = max(0.0, l["start"] - early, prev_end + 0.05)
        avail = max(0.25, min(nxt - 0.08, l["end"] + borrow, total) - start)
        need = avail / max(d, 1e-3)
        if need < 1.0: a = rb_stretch(a, sr, max(need, min_ratio))
        elif d < 0.75 * l["slot"]: a = rb_stretch(a, sr, min(l["slot"] * 0.9 / d, max_slow))
        over = max(0.0, len(a) / sr - avail)
        a = a[: int(avail * sr)]
        f = int(0.015 * sr)
        if len(a) > 2 * f: a[:f] *= np.linspace(0, 1, f); a[-f:] *= np.linspace(1, 0, f)
        s = int(start * sr); out[s: s + len(a)] += a[: len(out) - s]
        prev_end = start + len(a) / sr
        info.append(dict(need_ratio=need, overflow=over))
    return out[: int(total * sr)], info

def timing_stage(clips, lines_by, tts_by, cfg, tts_key):
    def run():
        res = {}
        for c in clips:
            n = c["name"]; d = f"{WORK}/out/e2e_dub/{_h(key)}"; os.makedirs(d, exist_ok=True); p = f"{d}/{n}.wav"
            L, Wv = lines_by[n], tts_by[n]
            need = [l["slot"] / max(dur(w), 1e-3) for l, w in zip(L, Wv) if w and os.path.exists(w)]
            if cfg["timing"] == "repo":
                segs = [dict(start=l["start"], end=l["end"], text=t) for l, t in zip(L, Wv)]
                auds = [dict(audio=load(w, RP.sr), sample_rate=RP.sr) if w and os.path.exists(w) else dict(audio=np.zeros(0, np.float32), sample_rate=RP.sr) for w in Wv]
                y = RP._align_and_stitch_segments(auds, segs, c["dur"]); save(p, y, RP.sr); over = np.nan
            else:
                y, info = fit_fixed(L, Wv, c["dur"]); save(p, y, 24000); over = float(sum(i["overflow"] for i in info if i))
            res[n] = dict(path=p, fit_outside_15=float(np.mean(np.abs(np.array(need) - 1) > 0.15)) if need else np.nan, overflow_s=over)
        return res
    key = tts_key + cfg["timing"]
    return cached_stage("e2e_timing", cfg["timing"], key, run)

def mix_fixed(dub_path, vocals_path, accomp_path, out, target_lufs=-23.0):
    """Dub gain-matched to the original vocal stem, background kept at 1.0 with ~-7 dB ducking, LUFS after the mix."""
    sr = 44100; meter = pyln.Meter(sr)
    dub, voc, acc = load(dub_path, sr), load(vocals_path, sr), load(accomp_path, sr)
    n = len(acc); dub = np.pad(dub, (0, max(0, n - len(dub))))[:n]
    try: dub = pyln.normalize.loudness(dub, meter.integrated_loudness(dub), meter.integrated_loudness(voc))
    except Exception: pass
    env = np.convolve(np.abs(dub), np.ones(int(0.05 * sr)) / int(0.05 * sr), mode="same")
    pres = np.convolve((env > 10 ** (-40 / 20)).astype(np.float32), np.ones(int(0.2 * sr)) / int(0.2 * sr), mode="same")
    mix = dub + acc * (1 - 0.55 * np.clip(pres, 0, 1))
    try: mix = pyln.normalize.loudness(mix, meter.integrated_loudness(mix), target_lufs)
    except Exception: pass
    peak = np.max(np.abs(mix)) + 1e-9
    if peak > 0.89: mix *= 0.89 / peak
    return save(out, mix, sr)

def mix_stage(clips, dub_by, cfg, timing_key):
    def run():
        res = {}
        for c in clips:
            n = c["name"]; v, a = separate(c); d = f"{WORK}/out/e2e_mix/{_h(key)}"; os.makedirs(d, exist_ok=True); p = f"{d}/{n}.wav"
            if cfg["mix"] == "repo":
                orig = f"{WORK}/out/sep/{n}/orig_mono_{RP.sr}.wav"
                if not os.path.exists(orig): sh(f'ffmpeg -y -v error -i "{c["path"]}" -ar {RP.sr} -ac 1 "{orig}"')
                y = RP._process_audio(load(dub_by[n]["path"], RP.sr), original_audio_path=Path(orig)); save(p, y, RP.sr)
            else:
                mix_fixed(dub_by[n]["path"], v, a, p)
            res[n] = p
        return res
    key = timing_key + cfg["mix"]
    return cached_stage("e2e_mix", cfg["mix"], key, run)

def lipsync_one(kind, video, audio, out):
    a16 = out.replace(".mp4", "_a16.wav"); save(a16, load(audio, 16000), 16000)
    if kind == "musetalk15":
        d = ensure_musetalk(); cfgp = f"{LOCAL}/mt_{_h(out)}.yaml"
        Path(cfgp).write_text(f'task_0:\n  video_path: "{video}"\n  audio_path: "{a16}"\n  result_name: "{Path(out).name}"\n')
        rd = f"{LOCAL}/mt_results"
        sh(f"{VENVS}/musetalk/bin/python -m scripts.inference --inference_config {cfgp} --result_dir {rd} "
           f"--unet_model_path models/musetalkV15/unet.pth --unet_config models/musetalkV15/musetalk.json "
           f"--version v15 --use_float16 --ffmpeg_path /usr/bin", cwd=d, timeout=5400)
        shutil.copy(f"{rd}/v15/{Path(out).name}", out)
    elif kind in ("latentsync15", "latentsync16"):
        ver = kind[-2:]; d = ensure_latentsync(ver)
        unet = "configs/unet/stage2_512.yaml" if ver == "16" else "configs/unet/stage2.yaml"
        sh(f"{VENVS}/latentsync/bin/python -m scripts.inference --unet_config_path {unet} --inference_ckpt_path checkpoints/latentsync_unet.pt "
           f"--inference_steps 20 --guidance_scale 1.5 --enable_deepcache --video_path {video} --audio_path {a16} --video_out_path {out}", cwd=d, timeout=7200)
    return out

def lips_stage(clips, mix_by, cfg, mix_key):
    def run():
        res = {}
        for c in clips:
            n = c["name"]; d = f"{WORK}/out/e2e_video/{_h(key)}"; os.makedirs(d, exist_ok=True); final = f"{d}/{n}.mp4"
            src_v = c["path"]
            if cfg["lips"] != "none":
                try:
                    src_v = lipsync_one(cfg["lips"], c["path"], mix_by[n], f"{d}/{n}_lips.mp4")
                except Exception as e:
                    print(f"lip sync {cfg['lips']} failed on {n}: {str(e)[-600:]}"); res[n] = None; continue
            sh(f'ffmpeg -y -v error -i "{src_v}" -i "{mix_by[n]}" -map 0:v:0 -map 1:a:0 -c:v libx264 -crf 18 -preset veryfast -c:a aac -b:a 192k "{final}"')
            res[n] = final
        return res
    key = mix_key + cfg["lips"]
    return cached_stage("e2e_lips", cfg["lips"], key, run, sum(c["dur"] for c in clips))
print("e2e pieces ready")

In [ ]:
# ---- 16. Lip-sync and visual metrics --------------------------------------------------------------
def syncnet_eval(video):
    """LSE-D = min distance, LSE-C = confidence (Prajwal et al. 2020 protocol, via syncnet_python)."""
    d = ensure_syncnet(); ref = _h(video); wd = f"{LOCAL}/sn_{ref}"
    py = f"{VENVS}/syncnet/bin/python"
    sh(f'{py} run_pipeline.py --videofile "{video}" --reference {ref} --data_dir {wd}', cwd=d, timeout=1800)
    r = sh(f'{py} run_syncnet.py --videofile "{video}" --reference {ref} --data_dir {wd}', cwd=d, timeout=1800)
    txt = r.stdout + r.stderr
    dist = [float(x) for x in re.findall(r"Min dist:\s*([\d.]+)", txt)]; conf = [float(x) for x in re.findall(r"Confidence:\s*([\d.]+)", txt)]
    off = [int(x) for x in re.findall(r"AV offset:\s*(-?\d+)", txt)]
    shutil.rmtree(wd, ignore_errors=True)
    if not conf: return dict(lse_c=np.nan, lse_d=np.nan, av_offset=np.nan)
    return dict(lse_c=float(np.mean(conf)), lse_d=float(np.mean(dist)), av_offset=float(np.mean(off)))

_FACE = {}
def visual_eval(src_video, out_video, every=5, max_frames=120):
    """Mouth-crop FID (source vs dub frames) and identity CSIM (FaceNet VGGFace2) on aligned frames."""
    import cv2
    from facenet_pytorch import MTCNN, InceptionResnetV1
    from torchmetrics.image.fid import FrechetInceptionDistance
    from PIL import Image
    if not _FACE:
        _FACE["det"] = MTCNN(keep_all=False, device="cuda"); _FACE["id"] = InceptionResnetV1(pretrained="vggface2").eval().cuda()
    def frames(p):
        cap, fs, k = cv2.VideoCapture(p), [], 0
        while len(fs) < max_frames:
            ok, f = cap.read()
            if not ok: break
            if k % every == 0: fs.append(Image.fromarray(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)))
            k += 1
        return fs
    A, B = frames(src_video), frames(out_video); n = min(len(A), len(B))
    fid = FrechetInceptionDistance(feature=2048).cuda(); cs, nm = [], 0
    for a, b in zip(A[:n], B[:n]):
        ba, _, la = _FACE["det"].detect(a, landmarks=True); bb, _, lb = _FACE["det"].detect(b, landmarks=True)
        if ba is None or bb is None: continue
        crops = []
        for img, box, lm in ((a, ba[0], la[0]), (b, bb[0], lb[0])):
            (x1, y1), (x2, y2) = lm[3], lm[4]; cx, cy = (x1 + x2) / 2, (y1 + y2) / 2; s = max(24, 1.6 * abs(x2 - x1))
            mouth = img.crop((int(cx - s / 2), int(cy - s / 2), int(cx + s / 2), int(cy + s / 2))).resize((128, 128))
            face = img.crop(tuple(int(v) for v in box)).resize((160, 160))
            crops.append((mouth, face))
        to_u8 = lambda im: torch.tensor(np.array(im)).permute(2, 0, 1)[None].cuda()
        fid.update(to_u8(crops[0][0]), real=True); fid.update(to_u8(crops[1][0]), real=False); nm += 1
        with torch.inference_mode():
            fe = [_FACE["id"](((torch.tensor(np.array(f)).permute(2, 0, 1).float() - 127.5) / 128.0)[None].cuda()) for _, f in crops]
        cs.append(float(torch.nn.functional.cosine_similarity(fe[0], fe[1]).item()))
    return dict(fid_mouth=float(fid.compute()) if nm >= 10 else np.nan, csim=float(np.mean(cs)) if cs else np.nan, n_faces=nm)
print("lip-sync metrics ready")

In [ ]:
# ---- 17. Ablation: baseline, then one change per row, then combined rows ----------------------------------
BASE = dict(asr="repo", diar=False, mt="seamless", tts="repo_edge", timing="repo", mix="repo", lips="none")
best_lips = P["lipsync"][1] if PROFILE != "free_t4" else P["lipsync"][0]
best_tts = "azure_ov" if "azure_ov" in TTS_SYSTEMS else "edge_ov"
ROWS = [
    ("baseline", {}),
    ("+ WhisperX ASR + sentence lines", dict(asr="whisperx")),
    ("+ WhisperX + diarization (per-speaker refs)", dict(asr="whisperx", diar=True)),
    ("+ TranslateGemma MT", dict(mt="tgemma")),
    ("+ MADLAD MT", dict(mt="madlad")),
    ("+ duration-aware MT (TranslateGemma + Gemma rewrite)", dict(mt="tgemma_dur")),
    ("+ edge-tts voice by gender + OpenVoice VC (Azure proxy)", dict(tts="edge_ov")),
    ("+ OmniVoice clone [NC]", dict(tts="omni")),
    ("+ OmniVoice clone + duration [NC]", dict(tts="omni_dur")),
    ("+ OmniVoice native carrier + OpenVoice VC [NC]", dict(tts="omni_native_ov")),
    ("+ timing fixes", dict(timing="fixed")),
    ("+ mix fixes", dict(mix="fixed")),
]
if RUN["lipsync"]:
    ROWS += [(f"+ {k}", dict(lips=k)) for k in P["lipsync"]]
if RUN["api_rows"]:
    if "gemini" in MT_SYSTEMS: ROWS += [("+ Gemini MT [API]", dict(mt="gemini")), ("+ Gemini duration-aware MT [API]", dict(mt="gemini_dur"))]
    if "azure_ov" in TTS_SYSTEMS: ROWS.append(("+ Azure hy-AM + OpenVoice VC [API]", dict(tts="azure_ov")))
    if "fish" in TTS_SYSTEMS: ROWS.append(("+ Fish S2.1 [API]", dict(tts="fish")))
ROWS += [
    ("BEST commercial combo", dict(asr="whisperx", diar=True, mt="tgemma_dur", tts=best_tts, timing="fixed", mix="fixed",
                                   lips=best_lips if RUN["lipsync"] else "none")),
    ("RESEARCH ceiling [NC]", dict(asr="whisperx", diar=True, mt="tgemma_dur", tts="omni_native_ov", timing="fixed", mix="fixed",
                                   lips=best_lips if RUN["lipsync"] else "none")),
]
if ROW_FILTER: ROWS = [r for r in ROWS if r[0] in ROW_FILTER]

COMMERCIAL_LIPS = dict(none=True, musetalk15=True, latentsync15="check detector", latentsync16="check detector")
def commercial(cfg):
    flags = [MT_COMMERCIAL[cfg["mt"]], TTS_COMMERCIAL[cfg["tts"]], COMMERCIAL_LIPS[cfg["lips"]]]
    if False in flags: return "no"
    return "yes" if all(f is True for f in flags) else "check"

E2E = []
clips = VIDEOS
if RUN["video"] and clips:
    for name, change in ROWS:
        cfg = {**BASE, **change}
        if cfg["tts"] not in TTS_SYSTEMS or cfg["mt"] not in MT_SYSTEMS: continue
        print(f"=== {name}: {cfg}")
        try:
            lines, k_asr = asr_stage(clips, cfg)
            refs = {c["name"]: speaker_refs(c, lines[c["name"]], _h(k_asr)) for c in clips}
            texts, k_mt = mt_stage(clips, lines, refs, cfg)
            tts, k_tts = tts_stage(clips, lines, texts, refs, cfg)
            dub, k_tim = timing_stage(clips, lines, tts, cfg, k_tts)
            mixd, k_mix = mix_stage(clips, dub, cfg, k_tim)
            vids, k_lip = lips_stage(clips, mixd, cfg, k_mix)
        except Exception as e:
            print(f"row failed: {str(e)[-1200:]}"); continue
        # voice / accent / style per line (dub line audio vs that speaker's held-out source audio)
        items = []
        for c in clips:
            n = c["name"]; v16 = load(separate(c)[0], 16000)
            for k, (l, t, w) in enumerate(zip(lines[n], texts[n], tts[n])):
                sp = f"{LOCAL}/src_{_h(n + str(k))}.wav"
                save(sp, v16[int(l["start"] * 16000): int(l["end"] * 16000)], 16000)
                items.append(dict(wav=w, text_hy=t, ref16_path=refs[n][l["speaker"]]["held16"], src16_path=sp, src_text_en=l["text"]))
        vdf = voice_eval(items); vs = summarize_voice(vdf)
        lip = []
        for c in clips:
            v = vids.get(c["name"])
            if not v: continue
            try: m = syncnet_eval(v)
            except Exception as e: print("SyncNet failed (no face track?):", str(e)[-300:]); m = dict(lse_c=np.nan, lse_d=np.nan, av_offset=np.nan)
            try: m.update(visual_eval(c["path"], v))
            except Exception as e: print("visual metrics failed:", str(e)[:200])
            lip.append(m)
        lipm = pd.DataFrame(lip).mean(numeric_only=True).to_dict() if lip else {}
        keys = [k_asr, k_mt, k_tts, k_tim, k_mix, k_lip]
        secs = sum(STAGE_PERF[k]["seconds"] for k in keys if k in STAGE_PERF)
        peak = max(max(STAGE_PERF[k].get("torch_peak_gb", 0), STAGE_PERF[k].get("smi_peak_gb", 0)) for k in keys if k in STAGE_PERF)
        E2E.append(dict(row=name, commercial_ok=commercial(cfg), **{f"cfg_{k}": v for k, v in cfg.items()},
                        n_lines=int(sum(len(lines[c["name"]]) for c in clips)),
                        n_speakers=float(np.mean([len(refs[c["name"]]) for c in clips])),
                        fit_outside_15=float(np.nanmean([dub[c["name"]]["fit_outside_15"] for c in clips])),
                        overflow_s=(np.nan if all(np.isnan(dub[c["name"]]["overflow_s"]) for c in clips)
                                    else float(np.nansum([dub[c["name"]]["overflow_s"] for c in clips]))),
                        **{f"v_{k}": v for k, v in vs.items()}, **lipm,
                        rtf=secs / sum(c["dur"] for c in clips), vram_peak_gb=peak))
        pd.DataFrame(E2E).to_csv(f"{WORK}/results/ablation_e2e_partial.csv", index=False)

# reference row: the original video with its original audio (upper bound for LSE, FID=0, CSIM=1)
if RUN["video"] and clips and RUN["lipsync"]:
    def _safe_sn(p):
        try: return syncnet_eval(p)
        except Exception: return dict(lse_c=np.nan, lse_d=np.nan, av_offset=np.nan)
    ref = pd.DataFrame([_safe_sn(c["path"]) for c in clips]).mean(numeric_only=True).to_dict()
    E2E.append(dict(row="REFERENCE original video + original audio", **ref))
E2E_TABLE = pd.DataFrame(E2E); display(E2E_TABLE)

In [ ]:
# ---- 18. Final ablation table: component metrics (FLEURS) + end-to-end metrics (video) --------------------
ASR_NAME = dict(repo="repo_whisper", whisperx="whisperx_turbo")
def comp(table, system, cols):
    if table is None or system not in set(table.system): return {c: np.nan for c in cols}
    r = table[table.system == system].iloc[0]; return {c: r.get(c, np.nan) for c in cols}

rows = []
for name, change in ROWS:
    cfg = {**BASE, **change}
    r = dict(row=name, change=", ".join(f"{k}={v}" for k, v in change.items()) or "-")
    r.update({f"asr_{k}": v for k, v in comp(ASR_TABLE, ASR_NAME[cfg["asr"]], ["wer", "cer"]).items()})
    r.update({f"mt_{k}": v for k, v in comp(MT_TABLE, cfg["mt"], ["chrfpp", "comet22", "fit_outside_15", "english_leak"]).items()})
    r.update({f"tts_{k}": v for k, v in comp(TTS_TABLE, cfg["tts"], ["secs_wavlm", "secs_ecapa", "utmos", "cer_hy", "p_hy", "ali", "rate_corr"]).items()})
    if len(E2E_TABLE) and name in set(E2E_TABLE.row):
        e = E2E_TABLE[E2E_TABLE.row == name].iloc[0]
        for c in ["commercial_ok", "n_speakers", "fit_outside_15", "overflow_s", "v_secs_wavlm", "v_utmos", "v_cer_hy", "v_p_hy", "v_ali",
                  "v_d_f0_st", "v_rate_corr", "lse_c", "lse_d", "av_offset", "fid_mouth", "csim", "rtf", "vram_peak_gb"]:
            r[f"e2e_{c}"] = e.get(c, np.nan)
    rows.append(r)
ABL = pd.DataFrame(rows)
ABL.to_csv(f"{WORK}/results/ablation.csv", index=False)
for t, n in [(ASR_TABLE, "asr"), (MT_TABLE, "mt"), (TTS_TABLE, "tts"), (E2E_TABLE, "e2e"), (FIT, "fit_matrix")]:
    if t is not None and len(t): t.to_csv(f"{WORK}/results/{n}.csv", index=False)
PERF_TABLE = pd.DataFrame(PERF); PERF_TABLE.to_csv(f"{WORK}/results/perf_stages.csv", index=False)
with open(f"{WORK}/results/ablation.md", "w") as f:
    f.write(f"# Ablation ({PROFILE}, {GPU_NAME})\n\n" + ABL.round(3).to_markdown(index=False) + "\n")
pd.set_option("display.max_columns", 80)
display(ABL.round(3))
print("Higher is better: chrF++, COMET, SECS, UTMOS, p_hy, rate_corr, LSE-C, CSIM. Lower is better: WER, CER, fit_outside_15, ALI, LSE-D, FID, RTF.")

In [ ]:
# ---- 19. Speed: RTF and VRAM peak per stage --------------------------------------------------------------
if len(PERF_TABLE):
    g = PERF_TABLE.dropna(subset=["seconds"]).groupby(["stage", "variant"]).agg(
        seconds=("seconds", "max"), audio_sec=("audio_sec", "max"), torch_peak_gb=("torch_peak_gb", "max"), smi_peak_gb=("smi_peak_gb", "max"))
    g["rtf"] = g.seconds / g.audio_sec
    display(g.round(3))

In [ ]:
# ---- 20. Human check lists (CSV for native Eastern Armenian reviewers) ------------------------------------
HUMAN = f"{WORK}/results/human"; os.makedirs(HUMAN, exist_ok=True)
MT_CHECK = ["adequacy_1to5", "fluency_1to5", "register_ok (formal/informal you)", "gender_agreement_ok", "names_ok",
            "numbers_dates_ok", "idiom_ok", "fits_mouth_time (y/n)", "notes"]
if MT_OUT:
    mt_rows = []
    for k, i in enumerate(FLEURS):
        sysl = list(MT_OUT); random.Random(k).shuffle(sysl)          # hide system identity
        for j, s in enumerate(sysl):
            mt_rows.append(dict(item=k, source_en=i["text_en"], reference_hy=i["ref_hy"], candidate=f"C{j}", text_hy=MT_OUT[s][k],
                                _system=s, **{c: "" for c in MT_CHECK}))
    df = pd.DataFrame(mt_rows); df.drop(columns="_system").to_csv(f"{HUMAN}/mt_checklist.csv", index=False)
    df[["item", "candidate", "_system"]].to_csv(f"{HUMAN}/mt_key_DO_NOT_SHARE.csv", index=False)

# MOS + ABX listening sheet over the FLEURS TTS outputs
ab = []
tts_dirs = {s: f"{WORK}/out/tts_fleurs/{s}" for s in TTS_SYSTEMS}
for k, i in enumerate(FLEURS):
    files = {s: f"{d}/{k:04d}.wav" for s, d in tts_dirs.items() if os.path.exists(f"{d}/{k:04d}.wav")}
    if "repo_edge" not in files: continue
    for s, f in files.items():
        if s == "repo_edge": continue
        a, b = (files["repo_edge"], f) if random.random() < 0.5 else (f, files["repo_edge"])
        ab.append(dict(item=k, source_speaker_clip=i["wav16"], A=a, B=b, mos_A_1to5="", mos_B_1to5="",
                       more_native_A_or_B="", closer_to_actor_voice_A_or_B="", closer_to_actor_delivery_A_or_B="", _sys=s))
if ab:
    d = pd.DataFrame(ab); d.drop(columns="_sys").to_csv(f"{HUMAN}/listening_mos_abx.csv", index=False)
    d[["item", "A", "B", "_sys"]].to_csv(f"{HUMAN}/listening_key_DO_NOT_SHARE.csv", index=False)
print("Human sheets in", HUMAN)
print("""MT reviewer check list:
 1. Meaning complete? (adequacy 1-5)   2. Natural spoken Eastern Armenian? (fluency 1-5)
 3. Formal vs informal "you" right for the scene?   4. Verb/adjective agreement with speaker gender?
 5. Names transliterated consistently?   6. Numbers, dates, units correct and spoken naturally?
 7. Idioms translated by meaning, not word by word?   8. Could an actor say it in the original time slot?
Listening: rate MOS 1-5 for A and B, then choose A/B for 'more native', 'closer to actor voice', 'closer to actor delivery'.""")

## Known problems
1. **Free session limits.** T4 sessions disconnect after a few hours of idle or total time. All stages cache to Drive, so re-run all cells to resume. Venvs live on local disk and are rebuilt each session (MuseTalk and LatentSync take about 10-20 min each).
2. **Gated models.** pyannote community-1, Gemma / TranslateGemma, and FLORES+ need you to accept their terms on Hugging Face with the same account as `HF_TOKEN`. Otherwise those rows fail and are skipped.
3. **No bf16 on T4.** Gemma 3, TranslateGemma, and T5/MADLAD can overflow in fp16. The free profile loads them in 4-bit with fp32 compute. That is slower, and quality may differ slightly from bf16.
4. **Dependency drift.** WhisperX, NeMo, pyannote 4, MuseTalk (mmcv 2.0.1 + torch 2.0.1 cu118), and LatentSync (torch 2.5.1) pin different stacks. If a venv build fails, the stage prints the error and the row is skipped. Pin versions in `VENV_SPECS` once a set works for you.
5. **YouTube downloads.** HDTF clips come from YouTube. Colab IPs are often blocked (bot check), and per-clip licenses vary. Use them only for internal evaluation, or put your own clips in `MY_CLIPS_DIR`.
6. **FLEURS is read speech.** ASR and MT component numbers are optimistic compared with TV dialogue (ArmBench-ASR reports a median movie WER of 61.87% for Armenian). FLEURS has no speaker IDs, so speaker diversity is approximate.
7. **FLEURS gender label mapping** (`0=male, 1=female`) is assumed, not verified. It is used only to balance the sample.
8. **UTMOSv2 is not validated on Armenian.** Use it to rank systems; the absolute MOS is not meaningful. Run the listening sheet for decisions.
9. **SyncNet bias.** LSE-C/D favor frontal, well-lit faces. Clips with no detected face give NaN. Very short clips are unstable. Scores on 25 fps re-encoded video can differ from the original frame rate.
10. **FID on mouth crops** uses at most about 120 frames per clip. It is noisy and biased at small N; use it to compare rows, not as an absolute number.
11. **Accent metric limits** are listed in the method cell above.
12. **OmniVoice and edge-tts are not commercial.** `edge_ov` is a stand-in for Azure + OpenVoice so the free profile can test the idea. Re-run with `api_rows=True` and Azure keys for the real commercial row.
13. **API rows** (Gemini, Azure, Fish) cost money, change over time, and their model ids or SDK signatures in this notebook are unverified. Check `GEMINI_MODEL` and `FISH_MODEL`.
14. **LatentSync face detector license.** LatentSync installs `insightface`; check the license of the detector weights it downloads before any commercial use.
15. **Frame rate.** All clips are converted to 25 fps for lip sync and metrics. TV masters at 23.976 or 29.97 fps need a final retime, which is not done here.
16. **Baseline faithfulness.** The baseline calls the repo code directly, so its bugs are included on purpose: the stretch clamp is ignored, short lines are deleted, background sits at 0.2, and lip sync is skipped (the repo's MuseTalk call never succeeds).
17. **Run time.** Every ablation row re-runs SyncNet and back-ASR. On T4 with 4-5 clips, expect several hours for all rows. Use `ROW_FILTER` to run a subset.